# 07 – Vascular features of all samples
This notebook describes the tumour vasculature of every sample. It uses the CD31 vessel segmentation of notebook 06
(after the artefact removal in part 5) and the annotated cells of 03c. The aim is to compare samples and look for
trends: primary tumours vs metastases, and changes within a patient. Absolute values depend on staining and
segmentation, so they are less informative than the differences between samples.

**Two kinds of features**

| Kind | Features | Source |
|---|---|---|
| **Vessel features** (main results) | vessel area, vessel length density, vessel width, distance of the tumour tissue to the nearest vessel (poorly supplied tissue), how evenly vessels are spread | vessel segmentation of 06 only |
| **Cell features** (with limitation) | pericyte coverage, proliferating (Ki67⁺) endothelial cells, HIF1A of tumour cells vs distance to vessels, immune cells next to vessels | 06 + annotated cells of 03c |

**Limitation of the cell features.** 03a removed every cell that overlapped its artefact mask by more than 5 %. That mask
also covered ordinary tissue around vessels: in an 800 µm window of ss2 it was 14 % of the tumour region and contained
42 % of the bright CD31 pixels. Endothelial and perivascular cells are therefore under-represented in the cell tables,
to a different extent in each sample (`pct_bright_cd31_in_03a_artefact_mask` in the vessel summary of 06). Cell
features are shown as trends and should be read together with that column.

| Part | Content | Output (`FIG_DIR`) |
|---|---|---|
| 1 | Settings, loading | – |
| 2 | Vessel features per sample | `vascular_features.csv`, `7-1_*` |
| 3 | Cell features per sample | `7-2_*` |
| 4 | Primary vs metastasis, within patients | `7-3_*`, `7-4_*` |
| 5 | Vessel maturity: αSMA coverage measured in the image (PDGFRβ shown), tissue around immature vs mature vessels; 4.1 features by vessel class | `7-5_*`, `maturity_*.csv`, `features_by_vessel_class*.csv`, `maturity_qc/` |

## 1. Settings and loading

In [ ]:
# ── EDIT ────────────────────────────────────────────────────────────────────────────────────────
OUTPUT_ROOT      = 'path/to/output'          # same as in 03a–06
SAMPLES          = [f'ss{i}' for i in range(1, 19)]         # samples without 06 output are skipped
VESSEL_DIR       = 'vessels'                                # OUT_NAME of notebook 06
ARTEFACT_GEOJSON = OUTPUT_ROOT + '/{s}/{s}_artefacts.geojson'   # artefact annotations of 06 part 5 (left out here too)
FIG_DIR          = OUTPUT_ROOT + '/vessels/features/'
PIXEL_SIZE_UM    = 0.5

POOR_SUPPLY_UM   = 100       # tissue farther than this from the nearest vessel counts as poorly supplied (O2 diffusion ~100–150 µm)
GRID_UM          = 250       # block size for the local vessel area (heterogeneity)
MIN_BLOCK_COVER  = 0.5       # blocks with less tumour region than this share are not used
NEAR_VESSEL_UM   = 10        # cells this close to a vessel (or inside) count as "next to a vessel" (as PERIVASCULAR_UM in 06)
DIST_BINS_UM     = [0, 20, 50, 100, 200]   # distance bins to the nearest vessel; the last bin is ≥ 200 µm (incl. > 256 µm)
KI67_Z           = 1.0       # a cell counts as Ki67⁺ if its Ki67 z-score is above this (as in 05)
HIF1A_Z          = 1.0       # a cell counts as HIF1A⁺ if its HIF1A z-score is above this (as POS_Z in 05)
MIN_CELLS        = 20        # groups with fewer cells in a sample are not summarised
SEED             = 0

SAMPLE_KEY = {  # CODEX sample -> transcriptomic name (as in 05)
    'ss1': 'S3-P',   'ss2': 'S3-M1',  'ss3': 'S4-P',   'ss4': 'S6-P',   'ss5': 'S7-M2',  'ss6': 'S9-M',
    'ss7': 'S12-P',  'ss8': 'S12-R',  'ss9': 'S12-M1', 'ss10': 'S12-M2', 'ss11': 'S2-P', 'ss12': 'S7-M1',
    'ss13': 'S11-P2', 'ss14': 'S1-M', 'ss15': 'S3-M2', 'ss16': 'S10-P2', 'ss17': 'S10-M', 'ss18': 'S13-M'}
ORDER = ['S1-M', 'S2-P', 'S3-P', 'S3-M1', 'S3-M2', 'S4-P', 'S6-P', 'S7-M1', 'S7-M2', 'S9-M',
         'S10-P2', 'S10-M', 'S11-P2', 'S12-P', 'S12-R', 'S12-M1', 'S12-M2', 'S13-M']
PATIENTS = {'Patient 3': ['S3-P', 'S3-M1', 'S3-M2'], 'Patient 7': ['S7-M1', 'S7-M2'],
            'Patient 10': ['S10-P2', 'S10-M'], 'Patient 12': ['S12-P', 'S12-R', 'S12-M1', 'S12-M2']}

# 03c cell type -> class (as CT_MAP in 05); unlisted types are printed and ignored for the class-based features
CT_MAP = {
    'Tumour': 'Tumour cells', 'Tumour (epithelial)': 'Tumour cells', 'VEGFR1+': 'Tumour cells',
    'Endothelial': 'Endothelial cells', 'Endothelial (weak)': 'Endothelial cells', 'VWF+': 'Endothelial cells',
    'Pericyte': 'Pericytes', 'Mural cell': 'Pericytes', 'SMA+': 'Smooth muscle cells', 'Smooth muscle': 'Smooth muscle cells',
    'ECM': 'Fibroblasts', 'CAF': 'Fibroblasts', 'Fibroblast': 'Fibroblasts', 'CD34+ CAF': 'Fibroblasts', 'CD34+ stromal cell': 'Fibroblasts',
    'Macrophage': 'Macrophages', 'CD68+ CD45-': 'Macrophages', 'Monocyte': 'Monocytes', 'CD14+ myeloid': 'Monocytes',
    'T cell': 'T cells', 'CD8 T cell': 'T cells', 'CD4-like T cell': 'T cells', 'Treg': 'T cells', 'B cell': 'B cells',
    'Epithelial': 'Epithelial cells', 'PanCK+ cell': 'Epithelial cells'}
MURAL_CLASSES  = ['Pericytes', 'Smooth muscle cells']       # cells that cover vessels
MACROPHAGE_CLASSES = ['Macrophages']                         # e.g. add 'Monocytes' to include them
NEIGHBOUR_CLASSES = ['Tumour cells', 'Fibroblasts', 'Pericytes', 'Macrophages', 'Monocytes', 'T cells', 'B cells']
# ────────────────────────────────────────────────────────────────────────────────────────────────

import os, json, warnings, logging
import numpy as np, pandas as pd, tifffile, cv2, anndata as ad
import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import ndimage as ndi
from scipy.stats import mannwhitneyu
warnings.filterwarnings('ignore'); logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({'font.family': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 11,
                     'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
                     'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
rng = np.random.default_rng(SEED)
COND_COL = {'primary': '#4c72b0', 'recurrence': '#8172b3', 'metastasis': '#dd8452'}

def save(fig, name, show=True):
    fig.savefig(FIG_DIR + name + '.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show() if show else None
    plt.close(fig)

def condition_of(label):
    return {'P': 'primary', 'M': 'metastasis', 'R': 'recurrence'}.get(label.split('-')[-1][0], 'other')

def annotation_mask(path, shape, ds):
    """QuPath annotations (closed shapes or lines drawn around an area) rasterised on a grid down-sampled by ds;
    detections in the file are ignored (as in 06 part 5)."""
    m = np.zeros(shape, np.uint8)
    if not path or not os.path.exists(path):
        return m.astype(bool)
    g = json.load(open(path))
    feats = g['features'] if isinstance(g, dict) and 'features' in g else (g if isinstance(g, list) else [g])
    def rings(geom):
        t = geom.get('type') if isinstance(geom, dict) else None
        if t == 'GeometryCollection':
            for gg in geom.get('geometries', []): yield from rings(gg)
        elif t in ('Polygon', 'MultiPolygon'):
            for poly in ([geom['coordinates']] if t == 'Polygon' else geom['coordinates']):
                for k, r in enumerate(poly): yield np.asarray(r, float), k == 0
        elif t in ('LineString', 'MultiLineString'):
            for line in ([geom['coordinates']] if t == 'LineString' else geom['coordinates']):
                if len(line) >= 3: yield np.asarray(line, float), True
    for f in feats:
        if not isinstance(f, dict): continue
        p = f.get('properties') or {}
        cls = p.get('classification'); cls = cls.get('name', '') if isinstance(cls, dict) else str(cls or '')
        if str(p.get('objectType', 'annotation')).lower() != 'annotation' or cls == 'Vessel':
            continue
        for r, outer in rings(f.get('geometry', f)):
            cv2.fillPoly(m, [np.round(r / ds).astype(np.int32).reshape(-1, 1, 2)], 1 if outer else 0)
    return m.astype(bool)

def block_sum(a, b):
    """Sum of b x b blocks (the last partial row/column of blocks is dropped)."""
    h, w = a.shape[0] // b, a.shape[1] // b
    out = np.zeros((h, w), np.float64)
    for i in range(0, h, 64):
        j = min(i + 64, h)
        out[i:j] = a[i * b:j * b, :w * b].reshape(j - i, b, w, b).sum(axis=(1, 3))
    return out

summary06 = pd.read_csv(OUTPUT_ROOT + f'/{VESSEL_DIR}/vessels_summary.csv').set_index('sample')
S = {}
for s in SAMPLES:
    d = os.path.join(OUTPUT_ROOT, s, VESSEL_DIR) + os.sep
    if not os.path.exists(d + f'{s}_vessels.csv') or s not in summary06.index:
        print(f'{s}: no 06 results, skipped'); continue
    lab = SAMPLE_KEY.get(s, s.upper())
    S[lab] = dict(sample=s, dir=d, base=os.path.join(OUTPUT_ROOT, s, s + '_'), condition=condition_of(lab),
                  V=pd.read_csv(d + f'{s}_vessels.csv'), row=summary06.loc[s])
ORDER = [l for l in ORDER if l in S] + [l for l in S if l not in ORDER]
S = {l: S[l] for l in ORDER}
print(f'{len(S)} samples:', ', '.join(f"{l} ({d['condition'][0].upper()})" for l, d in S.items()))

## 2. Vessel features (from the segmentation only)
For each sample:
- **Vessel area**: CD31⁺ area in % of the tumour region (without the artefact areas).
- **Vessel length density**: total length of the vessel centre lines per mm² of tumour (mm/mm²). It does not depend on how
  many pieces a vessel falls into, so it is more robust than the number of objects.
- **Vessel width**: median width of the CD31⁺ area (each object weighted by its area).
- **Distance to the nearest vessel**: for every point of the tumour tissue (on a 2 µm grid). Summarised as the median
  distance and the share of tissue farther than `POOR_SUPPLY_UM` (poorly supplied tissue).
- **Heterogeneity**: the tumour is divided into `GRID_UM` blocks and the vessel area is computed per block. The
  coefficient of variation (SD / mean) across blocks is high when vessels are clustered in parts of the tumour, and low
  when they are spread evenly. The share of blocks without any vessel is also given.

In [ ]:
rows, DIST, BLOCKS = [], {}, {}
for lab, d in S.items():
    s, V = d['sample'], d['V']
    labels = tifffile.imread(d['dir'] + f'{s}_vessel_labels.tif')
    m = np.load(d['base'] + 'masks_lowres.npz'); ds = int(m['bg_ds'])
    region = m['region_lr'].astype(bool) if 'region_lr' in m else m['tissue_lr'].astype(bool)
    region &= ~annotation_mask(ARTEFACT_GEOJSON.format(s=s), region.shape, ds)
    h, w = region.shape
    ves_lr = np.zeros((h, w), bool)                                  # vessel mask on the 2 µm grid of the region mask
    hh, ww = min(h, labels.shape[0] // ds), min(w, labels.shape[1] // ds)
    for i in range(0, hh, 256):
        j = min(i + 256, hh)
        ves_lr[i:j, :ww] = (labels[i * ds:j * ds, :ww * ds] > 0).reshape(j - i, ds, ww, ds).any(axis=(1, 3))
    step = ds * PIXEL_SIZE_UM                                        # µm per low-resolution pixel
    dist = ndi.distance_transform_edt(~ves_lr) * step
    tissue = region & ~ves_lr
    dt = dist[tissue]
    DIST[lab] = dt[rng.choice(len(dt), min(len(dt), 200_000), replace=False)] if len(dt) else dt
    # heterogeneity: vessel area per block (full-resolution vessel pixels / region pixels)
    b = max(1, int(round(GRID_UM / step)))
    reg_b = block_sum(region.astype(np.float32), b)
    ves_full_b = np.zeros_like(reg_b)
    bf = b * ds
    for i in range(0, reg_b.shape[0]):
        rows_ = labels[i * bf:(i + 1) * bf, :reg_b.shape[1] * bf] > 0
        if rows_.shape[0] == bf:
            ves_full_b[i] = rows_.reshape(bf, reg_b.shape[1], bf).sum(axis=(0, 2))
    use = reg_b >= MIN_BLOCK_COVER * b * b
    frac = np.where(use, 100 * ves_full_b / np.maximum(reg_b * ds * ds, 1), np.nan)
    BLOCKS[lab] = frac
    fv = frac[use]
    region_mm2 = d['row']['region_mm2']
    wts = V.area_um2.to_numpy(); order = np.argsort(V.mean_width_um.to_numpy())
    cw = np.cumsum(wts[order]) / wts.sum() if len(V) else np.array([])
    rows.append(dict(label=lab, sample=s, condition=d['condition'], region_mm2=region_mm2,
                     vessel_area_pct=100 * V.area_um2.sum() / (region_mm2 * 1e6),
                     vessel_length_mm_per_mm2=V.skeleton_length_um.sum() / 1000 / region_mm2,
                     objects_per_mm2=len(V) / region_mm2,
                     width_um_median=V.mean_width_um.to_numpy()[order][np.searchsorted(cw, 0.5)] if len(V) else np.nan,
                     dist_to_vessel_um_median=np.median(dt) if len(dt) else np.nan,
                     poorly_supplied_pct=100 * (dt > POOR_SUPPLY_UM).mean() if len(dt) else np.nan,
                     block_cv=np.std(fv) / np.mean(fv) if len(fv) and np.mean(fv) > 0 else np.nan,
                     blocks_without_vessels_pct=100 * (fv == 0).mean() if len(fv) else np.nan,
                     pct_bright_cd31_in_03a_artefact_mask=d['row'].get('pct_bright_cd31_in_03a_artefact_mask', np.nan)))
    del labels, dist
F = pd.DataFrame(rows).set_index('label')
display(F.round(2)) if 'display' in globals() else print(F.round(2).to_string())

In [ ]:
VESSEL_FEATURES = {'vessel_area_pct': 'Vessel area (% of tumour)', 'vessel_length_mm_per_mm2': 'Vessel length density (mm/mm²)',
                   'width_um_median': 'Vessel width, median (µm)', 'dist_to_vessel_um_median': 'Distance to nearest vessel, median (µm)',
                   'poorly_supplied_pct': f'Tissue > {POOR_SUPPLY_UM} µm from a vessel (%)', 'block_cv': f'Heterogeneity (CV of {GRID_UM} µm blocks)'}

def sample_panels(table, features, name, title):
    n = len(features); cols = 2; rws = -(-n // cols)
    fig, axs = plt.subplots(rws, cols, figsize=(14, 3.3 * rws), squeeze=False)
    x = np.arange(len(table))
    for ax, (col, t) in zip(axs.flat, features.items()):
        ax.bar(x, table[col], color=[COND_COL.get(c, '0.6') for c in table.condition], width=0.75)
        ax.set_xticks(x); ax.set_xticklabels(table.index, rotation=60, ha='right', fontsize=9); ax.set_title(t, fontsize=11)
        ax.grid(axis='y', ls='--', color='0.85'); ax.set_axisbelow(True)
        if col.endswith('_rel_dist'): ax.axhline(1, color='k', lw=1, ls='--')   # 1 = same distance as the average cell
    for ax in list(axs.flat)[n:]: ax.axis('off')
    fig.legend(handles=[Line2D([0], [0], marker='s', ls='', markersize=10, color=c, label=k) for k, c in COND_COL.items()],
               loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(title, y=1.05, fontsize=13, fontweight='bold'); fig.tight_layout(); save(fig, name)

sample_panels(F, VESSEL_FEATURES, '7-1_vessel_features_per_sample', 'Vessel features per sample')

# distance of the tumour tissue to the nearest vessel: cumulative share of tissue within a distance
fig, ax = plt.subplots(figsize=(8, 5))
for lab, dt in DIST.items():
    if not len(dt): continue
    xs = np.sort(dt); ax.plot(xs, 100 * np.arange(1, len(xs) + 1) / len(xs), color=COND_COL.get(S[lab]['condition'], '0.5'), lw=1.4, alpha=0.85)
    ax.text(np.median(xs) + 1.5, 50, lab, fontsize=7, va='center', rotation=60)                 # label at the median distance
ax.axvline(POOR_SUPPLY_UM, color='0.4', ls='--', lw=1)
ax.set_xlim(0, 300); ax.set_ylim(0, 100); ax.set_xlabel('distance to the nearest vessel (µm)'); ax.set_ylabel('% of tumour tissue within this distance')
ax.legend(handles=[Line2D([0], [0], color=c, lw=2, label=k) for k, c in COND_COL.items()], loc='lower right', frameon=False)
ax.set_title('Tumour tissue vs distance to the nearest vessel'); fig.tight_layout(); save(fig, '7-1_distance_to_vessel_curves')

# local vessel area maps (heterogeneity)
n = len(BLOCKS); cols = 6; rws = -(-n // cols)
vmax = np.nanpercentile(np.concatenate([b[np.isfinite(b)] for b in BLOCKS.values() if np.isfinite(b).any()]), 98)
fig, axs = plt.subplots(rws, cols, figsize=(2.6 * cols, 3.0 * rws), squeeze=False)
for ax, (lab, bm) in zip(axs.flat, BLOCKS.items()):
    im = ax.imshow(bm, cmap='magma', vmin=0, vmax=vmax, interpolation='nearest'); ax.set_title(lab, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor('#dddddd')
    for sp in ax.spines.values(): sp.set_visible(False)
for ax in list(axs.flat)[n:]: ax.axis('off')
fig.colorbar(im, ax=axs.ravel().tolist(), shrink=0.6, label=f'vessel area (% of {GRID_UM} µm block)')
fig.suptitle(f'Local vessel area ({GRID_UM} µm blocks; grey = outside the tumour region)', fontweight='bold'); save(fig, '7-1_local_vessel_area_maps')

# vessel width distribution per sample (area-weighted: each object counts with its area)
fig, ax = plt.subplots(figsize=(12, 4.5))
for i, (lab, d) in enumerate(S.items()):
    V = d['V']
    if not len(V): continue
    q = np.repeat(V.mean_width_um.to_numpy(), np.maximum(1, np.round(V.area_um2.to_numpy() / 10).astype(int)))
    parts = ax.violinplot([q], positions=[i], widths=0.8, showmedians=True, showextrema=False)
    for pc in parts['bodies']: pc.set_facecolor(COND_COL.get(d['condition'], '0.6')); pc.set_alpha(0.6)
ax.set_xticks(range(len(S))); ax.set_xticklabels(list(S), rotation=60, ha='right'); ax.set_ylabel('vessel width (µm)')
ax.set_ylim(0, np.nanpercentile(np.concatenate([d['V'].mean_width_um for d in S.values()]), 99.5))
ax.set_title('Vessel width per sample (weighted by CD31⁺ area)'); fig.tight_layout(); save(fig, '7-1_vessel_width_per_sample')

## 3. Cell features (03c cells; see the limitation at the top)
Cells inside the drawn artefact areas are left out.
- **Pericyte coverage**: share of the CD31⁺ vessel area that belongs to vessel objects with at least one pericyte or
  smooth muscle cell inside or within `NEAR_VESSEL_UM`. Covered vessels are more mature; uncovered, immature vessels
  respond more to anti-VEGF therapy.
- **Proliferating endothelium**: share of endothelial cells with Ki67 z > `KI67_Z` (sprouting angiogenesis).
- **Distance of HIF1A⁺ and Ki67⁺ cells to the nearest vessel**: HIF1A⁺ cells (z > `HIF1A_Z`) and Ki67⁺ cells
  (z > `KI67_Z`), all cell types except endothelial cells (they lie on the vessels by definition).
  - The median distance of these cells is compared with the median distance of all cells. The ratio is above 1 when the
    marker⁺ cells lie farther from vessels than the average cell. For HIF1A this is the pattern expected for hypoxia.
  - The heatmaps show, per distance bin, the share of marker⁺ cells divided by the share of all cells (log2). Red:
    marker⁺ cells are over-represented at that distance.
- **Macrophages and vessels**:
  - *Macrophage-covered vessel area*: share of the CD31⁺ vessel area that belongs to vessel objects with at least one
    macrophage inside or within `NEAR_VESSEL_UM` (shown next to the pericyte coverage).
  - *Perivascular macrophages per mm vessel*: macrophages within `NEAR_VESSEL_UM` of a vessel per mm of vessel length,
    which does not depend on how many vessels a sample has.
  - *Distance of macrophages to the nearest vessel*: median, relative to all cells, and per distance bin (heatmap), as for
    the HIF1A⁺ and Ki67⁺ cells.
- **Cells next to vessels**: for each cell class, the share of its cells within `NEAR_VESSEL_UM` of a vessel, divided by
  that share for all cells (log2). Above 0: the class is enriched next to vessels.

In [ ]:
unmapped, CF, ENR, POSBIN, POSDIST = set(), [], {}, {'HIF1A⁺ cells': {}, 'Ki67⁺ cells': {}, 'Macrophages': {}}, {}
bins = DIST_BINS_UM + [np.inf]
bin_names = [f'{a}–{b}' for a, b in zip(DIST_BINS_UM[:-1], DIST_BINS_UM[1:])] + [f'≥ {DIST_BINS_UM[-1]}']
POS = {'HIF1A': HIF1A_Z, 'Ki67': KI67_Z}
for lab, d in S.items():
    s = d['sample']
    f = d['dir'] + f'{s}_cells_vessels.csv'
    if not os.path.exists(f):
        print(f'{lab}: no cell table'); continue
    C = pd.read_csv(f, dtype={0: str})                                 # cell names as text, as in the h5ad
    C = C.set_index(C.columns[0]); C.index = C.index.astype(str)
    if 'in_artefact_area' in C: C = C[~C.in_artefact_area.astype(bool)]
    C['cls'] = C.cell_type.map(CT_MAP)
    unmapped |= set(C.cell_type[C.cls.isna()])
    # Ki67 and HIF1A z-scores from 03c
    a = ad.read_h5ad(d['base'] + 'adata_annotated.h5ad', backed='r')
    var = list(a.var_names); idx = a.obs_names.astype(str).get_indexer(C.index)
    for mk in POS:
        if mk in var:
            col = a.X[:, var.index(mk)]
            col = np.asarray(col.toarray() if hasattr(col, 'toarray') else col, dtype=float).ravel()
            C[mk] = np.where(idx >= 0, col[np.clip(idx, 0, None)], np.nan)
    a.file.close()
    found = [mk for mk in POS if mk in C]
    print(f"{lab}: {len(C):,} cells, z-scores found for {100 * (idx >= 0).mean():.0f} % of them; markers: {found or 'none (failed in this sample?)'}")
    V = d['V']
    dist = C.dist_to_vessel_um.fillna(np.inf)                          # empty = farther than the 06 margin (> 256 µm)
    near = dist <= NEAR_VESSEL_UM                                      # inside a vessel (0) or within NEAR_VESSEL_UM
    mural = C[C.cls.isin(MURAL_CLASSES) & near]
    covered = set(mural.nearest_vessel_id[mural.nearest_vessel_id > 0])
    endo = C[C.cls == 'Endothelial cells']
    row = dict(label=lab, condition=d['condition'],
               pericyte_covered_area_pct=100 * V.area_um2[V.vessel_id.isin(covered)].sum() / V.area_um2.sum() if len(V) else np.nan,
               ki67_endothelial_pct=100 * (endo.Ki67 > KI67_Z).mean() if 'Ki67' in endo and len(endo) >= MIN_CELLS else np.nan,
               n_endothelial_cells=len(endo))
    # distance of marker+ cells (all cell types except endothelial cells) to the nearest vessel
    other = C.cls != 'Endothelial cells'
    d_all = dist[other]
    row['all_cells_dist_median_um'] = np.median(d_all) if len(d_all) else np.nan
    g_all = pd.cut(d_all, bins, right=False, labels=bin_names).value_counts(normalize=True)
    POSDIST[lab] = {'all cells': d_all.to_numpy()}
    for mk, z in POS.items():
        if mk not in C: continue
        pos = other & (C[mk] > z)
        d_pos = dist[pos]
        if len(d_pos) < MIN_CELLS: continue
        POSDIST[lab][mk + '⁺'] = d_pos.to_numpy()
        row[f'{mk.lower()}_pos_dist_median_um'] = np.median(d_pos)
        row[f'{mk.lower()}_pos_rel_dist'] = np.median(d_pos) / row['all_cells_dist_median_um'] if row['all_cells_dist_median_um'] > 0 else np.nan
        g_pos = pd.cut(d_pos, bins, right=False, labels=bin_names).value_counts(normalize=True)
        POSBIN[mk + '⁺ cells'][lab] = np.log2((g_pos + 1e-3) / (g_all + 1e-3)).where(g_all > 0)
    # macrophages and vessels
    mac = other & C.cls.isin(MACROPHAGE_CLASSES)
    macn = C[mac & near]
    mac_cov = set(macn.nearest_vessel_id[macn.nearest_vessel_id > 0])
    vlen_mm = V.skeleton_length_um.sum() / 1000
    row['macrophage_covered_area_pct'] = 100 * V.area_um2[V.vessel_id.isin(mac_cov)].sum() / V.area_um2.sum() if len(V) else np.nan
    row['perivascular_macrophages_per_mm_vessel'] = (mac & near).sum() / vlen_mm if vlen_mm > 0 and mac.sum() >= MIN_CELLS else np.nan
    if mac.sum() >= MIN_CELLS:
        d_mac = dist[mac]
        POSDIST[lab]['Macrophages'] = d_mac.to_numpy()
        row['macrophage_dist_median_um'] = np.median(d_mac)
        row['macrophage_rel_dist'] = np.median(d_mac) / row['all_cells_dist_median_um'] if row['all_cells_dist_median_um'] > 0 else np.nan
        g_mac = pd.cut(d_mac, bins, right=False, labels=bin_names).value_counts(normalize=True)
        POSBIN['Macrophages'][lab] = np.log2((g_mac + 1e-3) / (g_all + 1e-3)).where(g_all > 0)
    CF.append(row)
    share_all = near.mean()
    ENR[lab] = pd.Series({c: np.log2(max(near[C.cls == c].mean(), 1e-3) / share_all) if (C.cls == c).sum() >= MIN_CELLS and share_all > 0 else np.nan
                          for c in NEIGHBOUR_CLASSES})
CF = pd.DataFrame(CF).set_index('label') if CF else pd.DataFrame()
if unmapped: print('03c cell types not in CT_MAP (ignored):', sorted(unmapped))
F = F.join(CF.drop(columns='condition'), how='left') if len(CF) else F
F.to_csv(FIG_DIR + 'vascular_features.csv')
print('written:', FIG_DIR + 'vascular_features.csv')

CELL_FEATURES = {'pericyte_covered_area_pct': 'Pericyte-covered vessel area (%)', 'ki67_endothelial_pct': 'Ki67⁺ endothelial cells (%)',
                 'hif1a_pos_rel_dist': 'HIF1A⁺ cells: distance to vessel\n(median, relative to all cells)',
                 'ki67_pos_rel_dist': 'Ki67⁺ cells: distance to vessel\n(median, relative to all cells)',
                 'macrophage_covered_area_pct': 'Macrophage-covered vessel area (%)',
                 'perivascular_macrophages_per_mm_vessel': f'Macrophages within {NEAR_VESSEL_UM} µm\nper mm vessel length',
                 'macrophage_rel_dist': 'Macrophages: distance to vessel\n(median, relative to all cells)'}
CELL_FEATURES = {k: v for k, v in CELL_FEATURES.items() if k in F and F[k].notna().any()}
if CELL_FEATURES:
    sample_panels(F, CELL_FEATURES, '7-2_cell_features_per_sample', 'Cell features per sample (03c cells; see limitation)')

# median distance to the nearest vessel: all cells vs HIF1A+ and Ki67+ cells (endothelial cells left out)
if POSDIST:
    fig, ax = plt.subplots(figsize=(13, 4.5))
    mcol = {'all cells': '0.55', 'HIF1A⁺': '#d62728', 'Ki67⁺': '#2ca02c', 'Macrophages': '#ff7f0e'}
    for i, (lab, dd) in enumerate(POSDIST.items()):
        for k, v in dd.items():
            j = list(mcol).index(k)
            if len(v): ax.scatter(i + (j - 1.5) * 0.18, min(np.median(v), 260), color=mcol[k], s=40, zorder=3, edgecolor='k', lw=0.3)
    ax.set_xticks(range(len(POSDIST))); ax.set_xticklabels(list(POSDIST), rotation=60, ha='right')
    ax.set_ylabel('median distance to the nearest vessel (µm)'); ax.grid(axis='y', ls='--', color='0.85'); ax.set_axisbelow(True)
    ax.legend(handles=[Line2D([0], [0], marker='o', ls='', color=c, markeredgecolor='k', label=k) for k, c in mcol.items()], frameon=False, loc='upper left', bbox_to_anchor=(1.005, 1))
    ax.set_title('Distance of HIF1A⁺ cells, Ki67⁺ cells and macrophages to the nearest vessel (endothelial cells left out)')
    fig.tight_layout(); save(fig, '7-2_marker_cells_distance_to_vessel')

def heat(M, title, name, cmap, vmax, label, ax=None, fig=None):
    if M.empty: return
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(1.1 * M.shape[1] + 3, 0.38 * M.shape[0] + 1.6))
    im = ax.imshow(M.to_numpy(float), cmap=cmap, vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns, rotation=40, ha='right')
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels([f'{l} ({S[l]["condition"][0].upper()})' for l in M.index])
    ax.set_facecolor('#dddddd'); fig.colorbar(im, ax=ax, label=label, shrink=0.8); ax.set_title(title)
    if own: fig.tight_layout(); save(fig, name)

mk_tabs = {mk: pd.DataFrame(v).T.reindex(columns=bin_names) for mk, v in POSBIN.items() if v}
if mk_tabs:
    fig, axs = plt.subplots(1, len(mk_tabs), figsize=(7.2 * len(mk_tabs), 0.38 * len(S) + 2), squeeze=False)
    for ax, (mk, M) in zip(axs[0], mk_tabs.items()):
        heat(M, f'{mk} vs distance to the nearest vessel (µm)', None, 'RdBu_r', 1.5,
             f'log2 (share of {mk} / share of all cells)', ax=ax, fig=fig)
    fig.tight_layout(); save(fig, '7-2_marker_cells_vs_distance')
# vessel coverage: pericytes vs macrophages, and perivascular macrophages per mm vessel
if 'macrophage_covered_area_pct' in F:
    fig, axs = plt.subplots(1, 2, figsize=(16, 4.6), gridspec_kw={'width_ratios': [1.4, 1]})
    x = np.arange(len(F))
    axs[0].bar(x - 0.2, F['pericyte_covered_area_pct'], 0.4, color='#e7cc0d', label='pericytes / smooth muscle cells')
    axs[0].bar(x + 0.2, F['macrophage_covered_area_pct'], 0.4, color='#e10907', label='macrophages')
    axs[0].set_ylabel('% of CD31⁺ vessel area'); axs[0].set_title(f'Vessels with a pericyte or a macrophage within {NEAR_VESSEL_UM} µm')
    axs[0].set_ylim(0, 118); axs[0].set_yticks(range(0, 101, 20))
    axs[0].legend(frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(0.5, 1.0))
    axs[1].bar(x, F['perivascular_macrophages_per_mm_vessel'], color=[COND_COL.get(c, '0.6') for c in F.condition], width=0.75)
    axs[1].set_ylabel('macrophages per mm vessel length'); axs[1].set_title(f'Macrophages within {NEAR_VESSEL_UM} µm of a vessel')
    for ax in axs:
        ax.set_xticks(x); ax.set_xticklabels(F.index, rotation=60, ha='right', fontsize=9); ax.grid(axis='y', ls='--', color='0.85'); ax.set_axisbelow(True)
    axs[1].legend(handles=[Line2D([0], [0], marker='s', ls='', markersize=9, color=c, label=k) for k, c in COND_COL.items()], frameon=False)
    fig.tight_layout(); save(fig, '7-2_vessel_coverage_pericytes_macrophages')

E = pd.DataFrame(ENR).T.reindex(columns=NEIGHBOUR_CLASSES)
heat(E, f'Cells within {NEAR_VESSEL_UM} µm of a vessel: enrichment', '7-2_cells_next_to_vessels', 'RdBu_r', 2, 'log2 (share near vessels / all cells)')

## 4. Comparisons
### 4.1 Primary tumours vs metastases
Recurrences are left out, as in 05. p-values: Mann–Whitney U test, two-sided, not corrected for multiple testing. The
groups are small, so read them as trends.

In [ ]:
def boxes(ax, data, labels, colors, positions=None, width=0.6):
    positions = positions if positions is not None else np.arange(len(data))
    for x, dv, c in zip(positions, data, colors):
        dv = np.asarray([v for v in dv if np.isfinite(v)])
        if not len(dv): continue
        q1, med, q3 = np.percentile(dv, [25, 50, 75])
        ax.add_patch(plt.Rectangle((x - width / 2, q1), width, q3 - q1, facecolor=matplotlib.colors.to_rgba(c, 0.5), edgecolor=c, lw=1.6))
        ax.plot([x - width / 2, x + width / 2], [med, med], color=c, lw=2)
        ax.plot([x, x], [dv.min(), q1], color=c, lw=1.6); ax.plot([x, x], [q3, dv.max()], color=c, lw=1.6)
        ax.scatter(x + rng.uniform(-width / 4, width / 4, len(dv)), dv, s=16, color='#20203a', zorder=3, alpha=0.85)
    ax.set_xticks(positions); ax.set_xticklabels(labels)
    ax.set_xlim(min(positions) - 0.7, max(positions) + 0.7)

ALL_FEATURES = {**VESSEL_FEATURES, **{k: v for k, v in CELL_FEATURES.items() if k in F}}
P, M = F[F.condition == 'primary'], F[F.condition == 'metastasis']
n = len(ALL_FEATURES); cols = 4; rws = -(-n // cols)
fig, axs = plt.subplots(rws, cols, figsize=(3.4 * cols, 3.6 * rws), squeeze=False)
for ax, (col, t) in zip(axs.flat, ALL_FEATURES.items()):
    boxes(ax, [P[col], M[col]], [f'P (n={P[col].notna().sum()})', f'M (n={M[col].notna().sum()})'], [COND_COL['primary'], COND_COL['metastasis']])
    p = mannwhitneyu(P[col].dropna(), M[col].dropna()).pvalue if P[col].notna().sum() > 1 and M[col].notna().sum() > 1 else np.nan
    ax.set_title(t + (f'\np = {p:.2f}' if np.isfinite(p) else ''), fontsize=10); ax.grid(axis='y', ls='--', color='0.85'); ax.set_axisbelow(True)
for ax in list(axs.flat)[n:]: ax.axis('off')
fig.suptitle('Primary tumours (P) vs metastases (M)', fontweight='bold'); fig.tight_layout(); save(fig, '7-3_primary_vs_metastasis')

### 4.2 Within patients
Samples of the same patient in the order primary → recurrence → metastases. With one section per time point, a change
between samples can reflect regional differences within the tumour as well as change over time.

In [ ]:
pats = {p: [l for l in ls if l in F.index] for p, ls in PATIENTS.items()}
pats = {p: ls for p, ls in pats.items() if len(ls) >= 2}
if pats:
    n = len(ALL_FEATURES); cols = 4; rws = -(-n // cols)
    fig, axs = plt.subplots(rws, cols, figsize=(3.6 * cols, 3.4 * rws), squeeze=False)
    pcol = dict(zip(pats, ['#1b9e77', '#d95f02', '#7570b3', '#e7298a']))
    for ax, (col, t) in zip(axs.flat, ALL_FEATURES.items()):
        for p, ls in pats.items():
            ax.plot(range(len(ls)), F.loc[ls, col], marker='o', color=pcol[p], label=p)
            for i, l in enumerate(ls): ax.annotate(l.split('-')[-1], (i, F.loc[l, col]), fontsize=7, xytext=(3, 3), textcoords='offset points')
        ax.set_title(t, fontsize=10); ax.set_xticks([]); ax.grid(axis='y', ls='--', color='0.85')
    for ax in list(axs.flat)[n:]: ax.axis('off')
    fig.legend(handles=[Line2D([0], [0], marker='o', color=c, label=p) for p, c in pcol.items()], loc='upper center', ncol=len(pcol), frameon=False, bbox_to_anchor=(0.5, 1.03))
    fig.tight_layout(); save(fig, '7-4_within_patients')

## 5. Vessel maturity: pericyte coverage measured in the image
Mature vessels are covered by pericytes or smooth muscle cells; immature, newly formed vessels are not. This part measures
the coverage directly from the **αSMA and PDGFRβ signal of the raw image**, not from the 03c cells (many perivascular
cells were removed by 03a, see the limitation at the top). It then asks whether the tissue around immature vessels
differs from the tissue around mature vessels.

**5.1 Coverage of every vessel piece.** For each sample:
1. αSMA and PDGFRβ are read from the raw image and corrected with the background surface of 03a (`MURAL_BG`: αSMA
   20th percentile of 256-px tissue blocks, PDGFRβ 15th percentile of 384-px blocks, as `BG_PARAMS` in 03a), but
   without setting the 03a artefact pixels to zero (as CD31 in 06), smoothed (`MURAL_SMOOTH_UM`) and scaled to 0–255
   (`MURAL_SCALE_PCT` of the tumour region = 255). A pixel is positive above the Otsu threshold of the region (per
   sample and channel, as for CD31; pixels in the 03a artefact mask are left out of this calculation, as in 06), or
   above a fixed value (`MURAL_THRESHOLD`). Channels that failed in a sample are skipped.
2. The vessel objects of 06 are cut into **pieces**: the part of a vessel object inside one `PIECE_UM` grid square.
   A capillary network that 06 labels as one object therefore has many pieces, each with its own coverage.
3. **Coverage = share of the vessel outline with positive signal within `RING_UM`** (inside or outside the CD31⁺ area,
   because mural cells lie directly on the endothelium and their signal overlaps the CD31 edge), for each channel. The
   outline is the border of the vessel mask of 06 as it is: no dilation and no lumen filling, as in 06. Where a vessel
   is cut across and its lumen is open, the lumen side of the wall belongs to the outline too; the wall is thin, so it
   lies within `RING_UM` of the signal outside and is counted as covered when the outside is.
4. **Chance check.** The same measure is taken for tissue `CHANCE_BAND_UM` away from vessels: the share of these pixels
   with positive signal within `RING_UM`. This is the coverage a vessel would get by chance if the signal had nothing to
   do with vessels. The ratio (vessel outline / tissue) is the **enrichment**; it is printed for each channel and saved
   in `<sample>_maturity_channels.csv`.
5. A piece is **mature** if its **αSMA coverage** (`MATURITY_CHANNEL`) is at least `MATURE_MIN`, otherwise
   **immature**. αSMA is used because in the first run PDGFRβ was found on much of the tissue away from vessels in
   several samples (stroma or tumour cells; enrichment < 2), so that almost every vessel reached the threshold with
   PDGFRβ. PDGFRβ is still measured and shown (5.2, QC figures), but not used for the class. This corresponds to the
   usual vessel maturation measure, the share of vessels with αSMA⁺ mural cells.
6. Each piece also gets its mean width (CD31⁺ area / centre-line length, as in 06) and the vessel area of its
   `GRID_UM` block (local vessel density, as in part 2).
7. Each 03c cell within `NEAR_BINS_UM[-1]` of a vessel is assigned to the piece of its nearest vessel pixel.

The tables are written to `<sample>/<VESSEL_DIR>/<sample>_vessel_pieces.csv`, `<sample>_cells_pieces.csv` and
`<sample>_maturity_channels.csv`, with the settings in `<sample>_maturity_settings.json`. A later run reads them instead
of measuring again, unless a measurement setting (`MURAL_*`, `RING_UM`, `LUMEN_MAX_UM2`, `PIECE_UM`, `CHANCE_BAND_UM`,
`NEAR_BINS_UM`) has changed or `RECOMPUTE = True`. `MATURITY_CHANNEL`, `MATURE_MIN` and the settings of 5.2–5.6 can be
changed without measuring again. **Check `FIG_DIR/maturity_qc/`** first: one figure per sample with all pieces
coloured by class and two 400 µm windows (the most vessel area, and the most mature vessel length) with the αSMA and
PDGFRβ signal and the vessel outlines.

**5.2–5.6 Analysis** (one value per sample; cells and vessels are not tested against each other). A sample is used in
5.3–5.6 only if
- the αSMA signal is concentrated along the vessels (enrichment ≥ `MIN_ENRICHMENT`); otherwise αSMA⁺ tissue such as
  stroma could make a vessel "mature" by chance, and
- it has at least `MIN_PIECES` immature and `MIN_PIECES` mature pieces, so that the comparison does not rest on a few
  vessels.

The excluded samples and the reason are printed in 5.2.
- 5.2 Share of mature vessel length per sample, and primary tumours vs metastases.
- 5.3 Check of the classification: immature, sprouting vessels should have more proliferating (Ki67⁺) endothelial cells.
- 5.4 Tissue around immature vs mature vessels: share of HIF1A⁺ cells, Ki67⁺ cells and macrophages among the cells
  whose nearest vessel piece is immature, compared with mature. To compare like with like, the shares are computed per
  **stratum** (distance bin × low/high local vessel density) and averaged with the same weights for both classes. The
  result per sample is log2(share near immature / share near mature): above 0 = more around immature vessels. A value
  is only calculated if at least `MIN_POSITIVE` positive cells lie around immature and around mature pieces; the
  numbers are printed. Across samples: Wilcoxon signed-rank test (is the effect consistently ≠ 0?); primary vs
  metastasis: Mann–Whitney.
- 5.5 Share of positive cells vs distance, immature vs mature (median of the samples).
- 5.6 Sensitivity: the effects with other coverage thresholds and for small vessels only.

In [ ]:
# ── EDIT ────────────────────────────────────────────────────────────────────────────────────────
RAW_IMAGE        = 'path/to/raw/{s}.ome.tif'   # raw image of each sample, as RAW_IMAGE in 06
MURAL_CHANNELS   = ['aSMA', 'PDGFRb']              # mural markers (names as in the channel list of 03a)
MURAL_THRESHOLD  = 'otsu'      # 'otsu' (per sample and channel) or a value 0–255 of the scaled signal, the same for all samples
MURAL_SMOOTH_UM  = 0.5         # Gaussian smoothing of αSMA / PDGFRβ (as for CD31)
MURAL_SCALE_PCT  = 99.9        # this percentile of the tumour region becomes 255
RING_UM          = 3           # an outline pixel is covered if αSMA⁺ or PDGFRβ⁺ signal lies within this distance
LUMEN_MAX_UM2    = 0           # 0 = no lumen filling, as in 06; > 0: holes up to this size (µm²) are filled before outline and width
PIECE_UM         = 50          # vessels are cut into pieces along a grid of this size
MATURITY_CHANNEL = 'aSMA'      # channel that decides the class: 'aSMA', 'PDGFRb' or 'any' (αSMA or PDGFRβ)
MATURE_MIN       = 0.30        # a piece is mature if at least this share of its outline is covered by MATURITY_CHANNEL
SENSITIVITY_MIN  = [0.2, 0.3, 0.5]   # other thresholds for 5.6
SMALL_VESSEL_UM  = 10          # pieces narrower than this (mean width) count as small vessels (capillaries)
MIN_BORDER_PX    = 20          # pieces with a shorter outline (in px, e.g. at the edge of the region) are not classified
NEAR_BINS_UM     = [0, 20, 50, 100]   # distance bins; cells farther than the last value are not used
ENDO_TOUCH_UM    = 2           # endothelial cells this close to a piece belong to it (5.3)
MIN_STRATUM      = 10          # strata with fewer cells around immature or around mature pieces are not used
MIN_PIECES       = 10          # samples with fewer immature or fewer mature pieces are not used in 5.3–5.6
MIN_ENRICHMENT   = 2.0         # samples whose MATURITY_CHANNEL covers the vessel outline less than this many times as often
                               # as the tissue CHANCE_BAND_UM away from vessels are not used in 5.3–5.6
MIN_POSITIVE     = 10          # 5.4: positive cells needed around immature and around mature pieces for a sample's value
CHANCE_BAND_UM   = (20, 50)    # tissue at this distance from vessels gives the coverage expected by chance (check in 5.1)
RECOMPUTE        = False       # True: measure again even if tables of an earlier run with the same settings exist
MURAL_BG         = {'aSMA': (256, 20), 'PDGFRb': (384, 15)}   # background block (px) and percentile per channel, as BG_PARAMS in 03a
BG_BLOCK_PX, BG_QUANTILE, BG_SMOOTH = 256, 20, 1.0   # for channels not in MURAL_BG (03a default)
TILE_PX          = 2048        # tile size; lower it if memory runs out
# ────────────────────────────────────────────────────────────────────────────────────────────────
from scipy.stats import wilcoxon
from skimage.morphology import skeletonize
QC_DIR = FIG_DIR + 'maturity_qc/'; os.makedirs(QC_DIR, exist_ok=True)
CLASS_COL = {'immature': '#00b4d8', 'mature': '#ffd60a', 'unclassified': '#bbbbbb'}   # QC figures (dark background)
CLS_PLOT = {'immature': '#0096c7', 'mature': '#b8860b'}             # the same classes on white backgrounds (plots of 5.2–5.5)
TIER_COL = {'uncovered': '#00b4d8', 'PDGFRβ⁺ only': '#2a9d55', 'αSMA⁺': '#c2185b'}
CH_NAME = {'aSMA': 'αSMA', 'PDGFRb': 'PDGFRβ', 'any': 'αSMA or PDGFRβ'}
MAT_NAME = CH_NAME.get(MATURITY_CHANNEL, MATURITY_CHANNEL)

def measure_settings():
    """Settings that change the measurement of 5.1; if they differ from the earlier run, the sample is measured again."""
    return dict(MURAL_CHANNELS=list(MURAL_CHANNELS), MURAL_THRESHOLD=MURAL_THRESHOLD, MURAL_SMOOTH_UM=MURAL_SMOOTH_UM,
                MURAL_SCALE_PCT=MURAL_SCALE_PCT, MURAL_BG={k: list(v) for k, v in MURAL_BG.items()},
                BG=[BG_BLOCK_PX, BG_QUANTILE, BG_SMOOTH], RING_UM=RING_UM, LUMEN_MAX_UM2=LUMEN_MAX_UM2, PIECE_UM=PIECE_UM,
                CHANCE_BAND_UM=list(CHANCE_BAND_UM), NEAR_UM=NEAR_BINS_UM[-1], GRID_UM=GRID_UM, PIXEL_SIZE_UM=PIXEL_SIZE_UM)

def _channel_pages(tif, n):
    for cand in (list(tif.pages), [p for s_ in tif.series for p in s_.pages], list(tif.series[0].pages) if tif.series else []):
        if len(cand) == n:
            return cand
    raise AssertionError(f'cannot find {n} channel pages')

def read_plane(path, names, name):
    with tifffile.TiffFile(path) as tif:
        return _channel_pages(tif, len(names))[names.index(name)].asarray()

def tiles(H, W, t=TILE_PX):
    for y in range(0, H, t):
        for x in range(0, W, t):
            yield y, min(y + t, H), x, min(x + t, W)

def window_mask(lr, ds, y0, y1, x0, x1):
    """Full-resolution window of a low-resolution mask."""
    ly0, lx0 = y0 // ds, x0 // ds
    m = lr[ly0:-(-y1 // ds), lx0:-(-x1 // ds)].repeat(ds, 0).repeat(ds, 1)
    m = m[y0 - ly0 * ds:, x0 - lx0 * ds:][:y1 - y0, :x1 - x0]
    out = np.zeros((y1 - y0, x1 - x0), bool); out[:m.shape[0], :m.shape[1]] = m
    return out

def pool_mean(plane, f, h, w):
    out = np.zeros((h, w), np.float32)
    hh, ww = min(h, plane.shape[0] // f), min(w, plane.shape[1] // f)
    for i in range(0, hh, 256):
        j = min(i + 256, hh)
        out[i:j, :ww] = plane[i * f:j * f, :ww * f].astype(np.float32).reshape(j - i, f, ww, f).mean(axis=(1, 3))
    return out

def background_grid(plane, tissue_lr, ds, block=BG_BLOCK_PX, q=BG_QUANTILE):
    """As in 03a / 06: q-th percentile of the tissue in blocks of `block` px, gaps filled, smoothed."""
    small = pool_mean(plane, ds, *tissue_lr.shape)
    b = max(1, block // ds)
    nby, nbx = -(-small.shape[0] // b), -(-small.shape[1] // b)
    pad = np.full((nby * b, nbx * b), np.nan, np.float32)
    pad[:small.shape[0], :small.shape[1]] = np.where(tissue_lr, small, np.nan)
    v = pad.reshape(nby, b, nbx, b).transpose(0, 2, 1, 3).reshape(nby, nbx, b * b)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        grid = np.nanpercentile(v, q, axis=2)
    grid[np.isfinite(v).mean(axis=2) < 0.10] = np.nan
    w = np.isfinite(grid).astype(np.float32)
    if w.sum() == 0:
        return np.zeros_like(grid, np.float32), b * ds
    g = np.nan_to_num(grid).astype(np.float32); out = np.full(g.shape, np.nan, np.float32)
    for sg in (max(BG_SMOOTH, 0.5), 4 * max(BG_SMOOTH, 0.5) + 4, 50.0):
        num = ndi.gaussian_filter(g * w, sg, mode='nearest'); den = ndi.gaussian_filter(w, sg, mode='nearest')
        todo = np.isnan(out) & (den > 1e-3); out[todo] = (num / np.maximum(den, 1e-6))[todo]
    out[np.isnan(out)] = np.nanmedian(grid)
    return out, b * ds

def surface_window(grid, block, y0, y1, x0, x1):
    def axis(a0, a1, n_in):
        c = np.clip((np.arange(a0, a1) + 0.5) / block - 0.5, 0, n_in - 1)
        i0 = np.floor(c).astype(np.int64); return i0, np.minimum(i0 + 1, n_in - 1), (c - i0).astype(np.float32)
    ya, yb, wy = axis(y0, y1, grid.shape[0]); xa, xb, wx = axis(x0, x1, grid.shape[1])
    gx = grid[:, xa] * (1 - wx) + grid[:, xb] * wx
    return gx[ya] * (1 - wy)[:, None] + gx[yb] * wy[:, None]

MURAL_SIGMA = MURAL_SMOOTH_UM / PIXEL_SIZE_UM

def mural_raw(MP, i, y0, y1, x0, x1, valid):
    """Background-corrected, smoothed mural channel i of a window; 0 outside the region (before smoothing)."""
    a = MP['planes'][i][y0:y1, x0:x1].astype(np.float32)
    a -= surface_window(*MP['bgs'][i], y0, y1, x0, x1); np.clip(a, 0, None, out=a)
    a[~valid] = 0
    return cv2.GaussianBlur(a, (0, 0), MURAL_SIGMA) if MURAL_SIGMA > 0 else a

def mural_window(MP, i, y0, y1, x0, x1, valid):
    return np.clip(mural_raw(MP, i, y0, y1, x0, x1, valid) * (255.0 / MP['scales'][i]), 0, 255)

def prepare_mural(s, base, m, region_lr):
    """Mural planes of one sample, their background surfaces, scales and thresholds."""
    cfg = json.load(open(base + 'run_config.json'))
    names, failed = cfg['channel_names'], set(cfg.get('FAILED_MARKERS', []))
    chans = [c for c in MURAL_CHANNELS if c in names and c not in failed]
    if not chans:
        return None
    ds = int(m['bg_ds'])
    planes = [read_plane(RAW_IMAGE.format(s=s), names, c) for c in chans]
    bg_tissue = m['tissue_lr'].astype(bool) & ~m['art_lr'].astype(bool)          # as in 03a / 06
    MP = dict(chans=chans, planes=planes,
              bgs=[background_grid(p, bg_tissue, ds, *MURAL_BG.get(c, (BG_BLOCK_PX, BG_QUANTILE))) for p, c in zip(planes, chans)])
    H, W = planes[0].shape
    stats_lr = region_lr & ~m['art_lr'].astype(bool)      # as in 06: bright folds of the 03a mask must not set the scale / threshold
    vals = [[] for _ in chans]
    for y0, y1, x0, x1 in tiles(H, W):                                           # every 8th pixel of the region
        v = window_mask(region_lr, ds, y0, y1, x0, x1)
        vs = window_mask(stats_lr, ds, y0, y1, x0, x1)
        if not vs[::8, ::8].any(): continue
        for i in range(len(chans)):
            vals[i].append(mural_raw(MP, i, y0, y1, x0, x1, v)[::8, ::8][vs[::8, ::8]])
    vals = [np.concatenate(v) for v in vals]
    MP['scales'] = [max(float(np.percentile(v, MURAL_SCALE_PCT)), 1e-6) for v in vals]
    MP['thr'], MP['pct_pos'] = [], []
    for v, sc in zip(vals, MP['scales']):
        s8 = np.clip(v * (255.0 / sc), 0, 255).astype(np.uint8)
        t = cv2.threshold(s8.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0] if MURAL_THRESHOLD == 'otsu' else float(MURAL_THRESHOLD)
        MP['thr'].append(float(t)); MP['pct_pos'].append(100 * float((s8 > t).mean()))
    return MP

def fill_lumen(mask, max_px):
    """Vessel mask with holes up to max_px filled (holes touching the window border are not filled)."""
    n, lab_, st, _ = cv2.connectedComponentsWithStats((~mask).astype(np.uint8), connectivity=4)
    hole = st[:, cv2.CC_STAT_AREA] <= max_px
    x0, y0 = st[:, cv2.CC_STAT_LEFT], st[:, cv2.CC_STAT_TOP]
    x1, y1 = x0 + st[:, cv2.CC_STAT_WIDTH], y0 + st[:, cv2.CC_STAT_HEIGHT]
    hole &= (x0 > 0) & (y0 > 0) & (x1 < mask.shape[1]) & (y1 < mask.shape[0])
    return mask | hole[lab_]

def maturity_coverage(p):
    """Coverage that decides the class (MATURITY_CHANNEL; 'any' = αSMA or PDGFRβ). NaN if the channel failed."""
    col = 'coverage' if MATURITY_CHANNEL == 'any' else f'coverage_{MATURITY_CHANNEL}'
    return p[col] if col in p else pd.Series(np.nan, index=p.index)

def classify(p, thr):
    """Maturity class of each piece for a coverage threshold."""
    c = pd.Series('unclassified', index=p.index, dtype=object)
    cov, ok = maturity_coverage(p), p.border_px >= MIN_BORDER_PX
    c[ok & (cov >= thr)] = 'mature'; c[ok & (cov < thr)] = 'immature'
    return c

def tier(p, thr):
    """uncovered / PDGFRβ⁺ only / αSMA⁺ (needs both channels)."""
    if not {'coverage_aSMA', 'coverage_PDGFRb'} <= set(p.columns):
        return None
    t = pd.Series('uncovered', index=p.index, dtype=object)
    t[p.coverage_PDGFRb >= thr] = 'PDGFRβ⁺ only'; t[p.coverage_aSMA >= thr] = 'αSMA⁺'
    t[p.border_px < MIN_BORDER_PX] = np.nan
    return t

def measure_maturity(lab, d):
    """Pieces with mural coverage and the piece of every cell near a vessel (read from file if measured before)."""
    s, out = d['sample'], d['dir']
    f_p, f_c, f_ch = out + f'{s}_vessel_pieces.csv', out + f'{s}_cells_pieces.csv', out + f'{s}_maturity_channels.csv'
    f_set = out + f'{s}_maturity_settings.json'
    same = os.path.exists(f_set) and json.load(open(f_set)) == json.loads(json.dumps(measure_settings()))
    if not RECOMPUTE and same and all(os.path.exists(f) for f in (f_p, f_c, f_ch)):
        p = pd.read_csv(f_p, index_col='piece'); c = pd.read_csv(f_c, dtype={'cell': str}).set_index('cell')
        ch_tab = pd.read_csv(f_ch, index_col='channel')
        print(f'{lab}: read {len(p):,} pieces and {len(c):,} cells from the earlier run'); chance_check(lab, ch_tab)
        return p, c, ch_tab
    if not RECOMPUTE and os.path.exists(f_p):
        print(f'{lab}: measurement settings changed since the earlier run (or not recorded), measuring again')
    m = np.load(d['base'] + 'masks_lowres.npz'); ds = int(m['bg_ds'])
    region_lr = (m['region_lr'] if 'region_lr' in m else m['tissue_lr']).astype(bool)
    region_lr &= ~annotation_mask(ARTEFACT_GEOJSON.format(s=s), region_lr.shape, ds)
    if not os.path.exists(RAW_IMAGE.format(s=s)):
        print(f'{lab}: raw image not found ({RAW_IMAGE.format(s=s)}), skipped'); return None, None, None
    MP = prepare_mural(s, d['base'], m, region_lr)
    if MP is None:
        print(f'{lab}: none of {MURAL_CHANNELS} available (failed), skipped'); return None, None, None
    labels = tifffile.imread(out + f'{s}_vessel_labels.tif'); H, W = labels.shape
    vmax = int(labels.max()) + 1
    seg = int(round(PIECE_UM / PIXEL_SIZE_UM)); ngx = -(-W // seg)
    rr, lumen_px = int(round(RING_UM / PIXEL_SIZE_UM)), LUMEN_MAX_UM2 / PIXEL_SIZE_UM ** 2
    disk = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * rr + 1, 2 * rr + 1))
    margin = int(np.ceil(max(NEAR_BINS_UM[-1], CHANCE_BAND_UM[1]) / PIXEL_SIZE_UM)) + 32
    cb0, cb1 = CHANCE_BAND_UM[0] / PIXEL_SIZE_UM, CHANCE_BAND_UM[1] / PIXEL_SIZE_UM
    chance = {ch: [0, 0] for ch in MP['chans'] + ['any']}          # [band pixels with mural signal within RING_UM, band pixels]
    # cells (positions only; types and markers are added in 5.2-5.6)
    C = pd.read_csv(out + f'{s}_cells_vessels.csv', dtype={0: str}); C = C.set_index(C.columns[0]); C.index = C.index.astype(str)
    if 'in_artefact_area' in C: C = C[~C.in_artefact_area.astype(bool)]
    cx_ = np.clip(np.round(C.x.to_numpy()).astype(int), 0, W - 1); cy_ = np.clip(np.round(C.y.to_numpy()).astype(int), 0, H - 1)
    acc, cells = [], []
    for ty0, ty1, tx0, tx1 in tiles(H, W):
        wy0, wy1, wx0, wx1 = max(ty0 - margin, 0), min(ty1 + margin, H), max(tx0 - margin, 0), min(tx1 + margin, W)
        lw = labels[wy0:wy1, wx0:wx1]
        if not lw.any(): continue
        mask = lw > 0
        filled = fill_lumen(mask, lumen_px) if lumen_px > 0 else mask
        dist, (iy, ix) = ndi.distance_transform_edt(~mask, return_indices=True)
        core = (slice(ty0 - wy0, ty1 - wy0), slice(tx0 - wx0, tx1 - wx0))
        iyc, ixc = iy[core], ix[core]; del iy, ix
        # piece of the nearest vessel pixel: (grid square of that pixel, vessel id)
        key = (((iyc + wy0) // seg).astype(np.int64) * ngx + (ixc + wx0) // seg) * vmax + lw[iyc, ixc]
        valid = window_mask(region_lr, ds, wy0, wy1, wx0, wx1)
        mc, fc, dc = mask[core], filled[core], dist[core]
        border = (filled & ~cv2.erode(filled.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool))[core] & valid[core]
        sk = skeletonize(filled)[core]
        yy, xx = np.nonzero(mc)
        cols = {'area_px': pd.Series(key[mc]).value_counts(), 'filled_px': pd.Series(key[fc]).value_counts(),
                'skel_px': pd.Series(key[sk]).value_counts(), 'border_px': pd.Series(key[border]).value_counts(),
                'sum_y': pd.Series(yy + ty0, index=key[mc]).groupby(level=0).sum(),
                'sum_x': pd.Series(xx + tx0, index=key[mc]).groupby(level=0).sum()}
        band = (dc >= cb0) & (dc <= cb1) & valid[core]                    # tissue away from vessels (chance level)
        anyp = np.zeros(border.shape, bool)
        for i, ch in enumerate(MP['chans']):
            pos = (mural_window(MP, i, wy0, wy1, wx0, wx1, valid) > MP['thr'][i]).astype(np.uint8)
            near = cv2.dilate(pos, disk).astype(bool)[core]                  # mural signal within RING_UM
            cols[f'pos_{ch}'] = pd.Series(key[border & near]).value_counts(); anyp |= near
            chance[ch][0] += int((band & near).sum()); chance[ch][1] += int(band.sum())
        cols['pos_any'] = pd.Series(key[border & anyp]).value_counts()
        chance['any'][0] += int((band & anyp).sum()); chance['any'][1] += int(band.sum())
        acc.append(pd.DataFrame(cols).fillna(0))
        sel = np.flatnonzero((cy_ >= ty0) & (cy_ < ty1) & (cx_ >= tx0) & (cx_ < tx1))
        if len(sel):
            r, q = cy_[sel] - ty0, cx_[sel] - tx0
            dd = dc[r, q] * PIXEL_SIZE_UM
            ok = dd <= NEAR_BINS_UM[-1]
            cells.append(pd.DataFrame({'dist_um': dd[ok], 'piece': key[r, q][ok]}, index=C.index[sel][ok]))
        del dist, key, dc
    p = pd.concat(acc).groupby(level=0).sum(); p = p[p.area_px > 0]
    p.index.name = 'piece'
    p['vessel_id'] = (p.index % vmax).astype(int)
    p['y'], p['x'] = p.sum_y / p.area_px, p.sum_x / p.area_px
    p['area_um2'] = p.area_px * PIXEL_SIZE_UM ** 2
    p['length_um'] = np.maximum(p.skel_px, 1) * PIXEL_SIZE_UM
    p['width_um'] = p.filled_px * PIXEL_SIZE_UM ** 2 / p.length_um
    p['coverage'] = p.pos_any / p.border_px.replace(0, np.nan)
    for ch in MP['chans']:
        p[f'coverage_{ch}'] = p[f'pos_{ch}'] / p.border_px.replace(0, np.nan)
    # local vessel density: vessel area in % of the tumour region of the GRID_UM block (as in part 2)
    bf = int(round(GRID_UM / PIXEL_SIZE_UM)); blr = max(1, bf // ds)
    reg_b = block_sum(region_lr.astype(np.float32), blr) * ds * ds
    by, bx = (p.y // bf).astype(int).clip(0, reg_b.shape[0] - 1), (p.x // bf).astype(int).clip(0, reg_b.shape[1] - 1)
    ves_b = p.area_px.groupby([by.to_numpy(), bx.to_numpy()]).sum()
    p['local_vessel_area_pct'] = 100 * ves_b.reindex(pd.MultiIndex.from_arrays([by, bx])).to_numpy() / np.maximum(reg_b[by, bx], 1)
    p = p.drop(columns=['sum_y', 'sum_x'])
    c = pd.concat(cells) if cells else pd.DataFrame(columns=['dist_um', 'piece']); c.index.name = 'cell'
    # per channel: threshold, outline coverage of the whole sample and the coverage expected by chance
    ch_tab = pd.DataFrame({'threshold_0_255': MP['thr'] + [np.nan], 'pct_region_above': MP['pct_pos'] + [np.nan]},
                          index=pd.Index(MP['chans'] + ['any'], name='channel'))
    ch_tab['outline_coverage_pct'] = [100 * p[f'pos_{ch}'].sum() / max(p.border_px.sum(), 1) for ch in ch_tab.index]
    ch_tab['chance_coverage_pct'] = [100 * chance[ch][0] / max(chance[ch][1], 1) for ch in ch_tab.index]
    ch_tab['enrichment'] = ch_tab.outline_coverage_pct / ch_tab.chance_coverage_pct.clip(lower=0.1)   # chance floored at 0.1 %
    p.to_csv(f_p); c.to_csv(f_c); ch_tab.to_csv(f_ch)
    json.dump(measure_settings(), open(f_set, 'w'), indent=1)
    cls = classify(p, MATURE_MIN)
    if MATURITY_CHANNEL != 'any' and MATURITY_CHANNEL not in MP['chans']:
        print(f'{lab}: {MAT_NAME} failed in this sample, its vessels are not classified')
    print(f"{lab}: {', '.join(f'{ch} threshold {t:.0f}/255 ({q:.1f} % of region pixels above)' for ch, t, q in zip(MP['chans'], MP['thr'], MP['pct_pos']))}; "
          f"{len(p):,} pieces, mature {100 * p.length_um[cls == 'mature'].sum() / max(p.length_um[cls != 'unclassified'].sum(), 1):.0f} % "
          f"of the vessel length; {len(c):,} cells within {NEAR_BINS_UM[-1]} µm")
    chance_check(lab, ch_tab)
    qc_figure(lab, MP, labels, p, cls, seg, vmax, ngx, region_lr, ds)
    del MP, labels
    return p, c, ch_tab

def chance_check(lab, ch_tab):
    """Outline coverage vs coverage of tissue CHANCE_BAND_UM away from vessels (enrichment), per channel."""
    for ch, r in ch_tab.iterrows():
        if ch == 'any' and len(ch_tab) <= 2: continue                    # one channel: 'any' is the same
        txt = (f'    {CH_NAME.get(ch, ch)}: {r.outline_coverage_pct:.1f} % of the vessel outline covered vs {r.chance_coverage_pct:.1f} % '
               f'of the tissue {CHANCE_BAND_UM[0]}–{CHANCE_BAND_UM[1]} µm from vessels (enrichment {r.enrichment:.1f})')
        if ch == MATURITY_CHANNEL:
            txt += ' - decides the class'
            if not r.enrichment >= MIN_ENRICHMENT:
                txt += f' -> NOT concentrated along the vessels (< {MIN_ENRICHMENT:g}): sample not used in 5.3–5.6'
        elif not r.enrichment >= MIN_ENRICHMENT:
            txt += ' -> not concentrated along the vessels (shown only)'
        print(txt)

def specific(lab):
    """Is MATURITY_CHANNEL concentrated along the vessels of this sample (enrichment >= MIN_ENRICHMENT)?"""
    t = CHAN.get(lab)
    return t is not None and MATURITY_CHANNEL in t.index and t.at[MATURITY_CHANNEL, 'enrichment'] >= MIN_ENRICHMENT

def qc_figure(lab, MP, labels, p, cls, seg, vmax, ngx, region_lr, ds, size_um=400):
    """Left: all pieces of the sample (coloured by class). Middle: window with the most vessel area. Right: window with the
    most mature vessel length (if any). Windows: mural signal, vessel outlines coloured by class."""
    H, W = labels.shape; sz = int(size_um / PIXEL_SIZE_UM)
    gidx = [(p.y // sz).astype(int), (p.x // sz).astype(int)]
    wins = [('most vessel area', p.area_px.groupby(gidx).sum().idxmax())]
    mat = p.length_um.where(cls == 'mature', 0)
    if (mat > 0).any():
        gm = mat.groupby(gidx).sum().idxmax()
        wins.append(('most mature vessel length' if gm != wins[0][1] else 'most mature vessel length (same window)', gm))
    names = ' + '.join({'aSMA': 'αSMA (magenta)', 'PDGFRb': 'PDGFRβ (green)'}.get(c, c) for c in MP['chans'])
    fig, axs = plt.subplots(1, 1 + len(wins), figsize=(7.5 * (1 + len(wins)), 7.4),
                            gridspec_kw={'width_ratios': [1.15] + [1] * len(wins)})
    for c_, col in CLASS_COL.items():
        q = p[cls == c_]
        axs[0].scatter(q.x, q.y, s=np.clip(q.length_um / 8, 1, 30), color=col, label=c_, linewidths=0)
    for k, (what, (gy, gx)) in enumerate(wins, start=1):
        y0, x0 = gy * sz, gx * sz; y1, x1 = min(y0 + sz, H), min(x0 + sz, W)
        axs[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec='w', lw=1.2))
        axs[0].text(x0 + sz * 0.04, y0 + sz * 0.04, str(k), color='w', fontsize=11, fontweight='bold', va='top')
        valid = window_mask(region_lr, ds, y0, y1, x0, x1)
        rgb = np.zeros((y1 - y0, x1 - x0, 3), np.float32)
        for i, ch in enumerate(MP['chans']):
            a = mural_window(MP, i, y0, y1, x0, x1, valid) / 255
            if ch == 'PDGFRb': rgb[..., 1] += a                   # PDGFRβ green
            else: rgb[..., 0] += a; rgb[..., 2] += a              # αSMA magenta
        lw = labels[y0:y1, x0:x1]
        yy, xx = np.mgrid[y0:y1, x0:x1]
        key = ((yy // seg).astype(np.int64) * ngx + xx // seg) * vmax + lw
        kc = pd.Series(cls).reindex(key[lw > 0]).fillna('unclassified').to_numpy()
        cmap = np.zeros(lw.shape, np.int8); cmap[lw > 0] = np.select([kc == 'immature', kc == 'mature'], [1, 2], 3)
        axs[k].imshow(np.clip(rgb * 1.3, 0, 1), interpolation='nearest')
        for v_, c_ in [(1, 'immature'), (2, 'mature'), (3, 'unclassified')]:
            if (cmap == v_).any():
                axs[k].contour((cmap == v_).astype(float), [0.5], colors=[CLASS_COL[c_]], linewidths=0.9)
        axs[k].set_xticks([]); axs[k].set_yticks([])
        axs[k].set_title(f'{k}: {size_um} µm window with the {what}\n{names}; outlines = CD31⁺ vessels by class', fontsize=11)
    axs[0].set_xlim(0, W); axs[0].set_ylim(H, 0); axs[0].set_aspect('equal'); axs[0].set_xticks([]); axs[0].set_yticks([])
    axs[0].set_facecolor('0.15'); axs[0].legend(loc='lower right', fontsize=9, markerscale=2)
    axs[0].set_title(f'{lab}: vessel pieces ({PIECE_UM} µm), {MAT_NAME} coverage ≥ {MATURE_MIN:.0%} = mature', fontsize=11)
    fig.tight_layout(); save(fig, 'maturity_qc/' + lab + '_maturity_qc', show=False)

PIECES, CELLP, CHAN = {}, {}, {}
for lab, d in S.items():
    p_, c_, ch_ = measure_maturity(lab, d)
    if p_ is not None:
        PIECES[lab], CELLP[lab], CHAN[lab] = p_, c_, ch_
print(f'\n{len(PIECES)} samples; check the figures in {QC_DIR}')

### 5.2 Share of mature vessels per sample
Left: share of the vessel length (centre line) in mature pieces (αSMA coverage ≥ `MATURE_MIN`). Hatched bars: samples
in which αSMA is not concentrated along the vessels (enrichment < `MIN_ENRICHMENT`, see 5.1); their share is not
reliable, and they are left out of the primary vs metastasis comparison and of 5.3–5.6. Right (descriptive): the vessel
length split by channel, αSMA⁺, PDGFRβ⁺ only and uncovered; PDGFRβ is not used for the class. The mean coverage is the
αSMA-covered share of the whole vessel outline of the sample (no threshold).

Second figure (`7-5_maturity_mature_vs_immature`): the vessel length of each sample split into the two classes used in
5.3–5.6, mature (αSMA-covered, gold) and immature (the rest, blue); the number is the mature share.

In [ ]:
def fmt_p(p):
    return 'n.a.' if not np.isfinite(p) else ('< 0.001' if p < 0.001 else f'{p:.3f}' if p < 0.01 else f'{p:.2f}')

def n_pieces(p, thr, small_only=False):
    """Number of immature and mature pieces (small vessels only if small_only)."""
    if small_only: p = p[p.width_um < SMALL_VESSEL_UM]
    c = classify(p, thr)
    return int((c == 'immature').sum()), int((c == 'mature').sum())

def eligible(lab, thr=None, small_only=False):
    """Used in 5.3–5.6: maturity channel concentrated along the vessels and enough pieces in both classes."""
    return specific(lab) and min(n_pieces(PIECES[lab], MATURE_MIN if thr is None else thr, small_only)) >= MIN_PIECES

def length_share(p, mask):
    L = p.length_um[p.border_px >= MIN_BORDER_PX]
    return 100 * L[mask.reindex(L.index).fillna(False)].sum() / L.sum() if L.sum() > 0 else np.nan

pos_col = 'pos_any' if MATURITY_CHANNEL == 'any' else f'pos_{MATURITY_CHANNEL}'
MAT = []
for lab, p in PIECES.items():
    cls, tr = classify(p, MATURE_MIN), tier(p, MATURE_MIN)
    ni, nm = int((cls == 'immature').sum()), int((cls == 'mature').sum())
    enr = CHAN[lab].at[MATURITY_CHANNEL, 'enrichment'] if MATURITY_CHANNEL in CHAN[lab].index else np.nan
    why = [] if enr >= MIN_ENRICHMENT else [f'{MAT_NAME} not concentrated along the vessels (enrichment {enr:.1f})']
    if min(ni, nm) < MIN_PIECES: why.append(f'{ni} immature / {nm} mature pieces')
    row = dict(label=lab, condition=S[lab]['condition'],
               mature_length_pct=length_share(p, cls == 'mature') if pos_col in p else np.nan,
               mean_coverage_pct=100 * p[pos_col].sum() / max(p.border_px.sum(), 1) if pos_col in p else np.nan,
               mature_length_pct_small=length_share(p[p.width_um < SMALL_VESSEL_UM], cls[p.width_um < SMALL_VESSEL_UM] == 'mature')
                                       if pos_col in p else np.nan,
               n_immature_pieces=ni, n_mature_pieces=nm, maturity_enrichment=enr, specific=bool(enr >= MIN_ENRICHMENT),
               used_in_5_3_to_5_6=not why, not_used_because='; '.join(why))
    if tr is not None:
        for t in TIER_COL: row[f'{t} (% length)'] = length_share(p, tr == t)
    for ch, r in CHAN[lab].iterrows():
        if ch != 'any': row[f'{ch}_enrichment'] = r.enrichment       # outline coverage / chance coverage (5.1)
    MAT.append(row)
MAT = pd.DataFrame(MAT).set_index('label')
MAT.to_csv(FIG_DIR + 'maturity_per_sample.csv')
show = [c for c in MAT if c not in ('used_in_5_3_to_5_6', 'not_used_because', 'specific')]
display(MAT[show].round(1)) if 'display' in globals() else print(MAT[show].round(1).to_string())
print(f'class: {MAT_NAME} coverage ≥ {MATURE_MIN:.0%}')
notused = MAT[~MAT.used_in_5_3_to_5_6]
if len(notused):
    print('not used in 5.3–5.6:')
    for lab, r in notused.iterrows(): print(f'    {lab}: {r.not_used_because}')

fig, axs = plt.subplots(1, 2, figsize=(16, 4.6))
x = np.arange(len(MAT))
bars = axs[0].bar(x, MAT.mature_length_pct, color=[COND_COL.get(c, '0.6') for c in MAT.condition], width=0.75)
for b_, ok in zip(bars, MAT.specific):
    if not ok: b_.set_hatch('///'); b_.set_alpha(0.45)
axs[0].set_ylabel('% of vessel length'); axs[0].set_title(f'Mature vessels ({MAT_NAME} coverage ≥ {MATURE_MIN:.0%})')
axs[0].set_ylim(0, 118); axs[0].set_yticks(range(0, 101, 20))
axs[0].legend(handles=[Line2D([0], [0], marker='s', ls='', markersize=9, color=c, label=k) for k, c in COND_COL.items()]
              + [plt.Rectangle((0, 0), 1, 1, fc='w', ec='0.4', hatch='///', label=f'{MAT_NAME} not vessel-specific')],
              frameon=False, loc='upper left', ncol=4, fontsize=9)
tcols = [f'{t} (% length)' for t in TIER_COL]
if all(c in MAT for c in tcols):
    b = np.zeros(len(MAT))
    for t, col in TIER_COL.items():
        v = MAT[f'{t} (% length)'].fillna(0).to_numpy(); axs[1].bar(x, v, bottom=b, color=col, width=0.75, label=t); b += v
    axs[1].set_ylabel('% of vessel length'); axs[1].set_ylim(0, 101); axs[1].legend(frameon=False, loc='upper right', fontsize=9)
    axs[1].set_title(f'Coverage by channel (descriptive; the class uses {MAT_NAME})')
else:
    axs[1].axis('off')
for ax in axs:
    ax.set_xticks(x); ax.set_xticklabels(MAT.index, rotation=60, ha='right', fontsize=9); ax.grid(axis='y', ls='--', color='0.85'); ax.set_axisbelow(True)
fig.tight_layout(); save(fig, '7-5_maturity_per_sample')

# mature (αSMA-covered) vs immature (the rest) per sample
fig, ax = plt.subplots(figsize=(12, 4.8))
mat = MAT.mature_length_pct.fillna(0).to_numpy(); imm = np.where(MAT.mature_length_pct.notna(), 100 - mat, 0)
for vals, bottom, c_ in ((mat, 0, 'mature'), (imm, mat, 'immature')):
    bars = ax.bar(x, vals, bottom=bottom, color=CLS_PLOT[c_], width=0.75, edgecolor='white', lw=0.8)
    for b_, ok in zip(bars, MAT.specific):
        if not ok: b_.set_hatch('///'); b_.set_alpha(0.45)
for xi, v in zip(x, MAT.mature_length_pct):
    if np.isfinite(v): ax.text(xi, v + 1.5 if v < 12 else v / 2, f'{v:.0f}', ha='center', va='bottom' if v < 12 else 'center', fontsize=8.5, color='#222')
ax.set_xticks(x); ax.set_xticklabels(MAT.index, rotation=60, ha='right', fontsize=9)
ax.set_ylim(0, 100); ax.set_ylabel('% of vessel length'); ax.set_axisbelow(True)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=CLS_PLOT['mature'], label=f'mature ({MAT_NAME}-covered, coverage ≥ {MATURE_MIN:.0%})'),
                   plt.Rectangle((0, 0), 1, 1, fc=CLS_PLOT['immature'], label='immature (the rest)'),
                   plt.Rectangle((0, 0), 1, 1, fc='w', ec='0.4', hatch='///', label=f'{MAT_NAME} not vessel-specific (not used in 5.3–5.6)')],
          frameon=False, fontsize=9, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3)
ax.set_title('Mature vs immature vessels per sample', fontweight='bold', pad=28)
fig.tight_layout(); save(fig, '7-5_maturity_mature_vs_immature')

MCOLS = {'mature_length_pct': f'Mature vessels (% length, {MAT_NAME} ≥ {MATURE_MIN:.0%})',
         'mean_coverage_pct': f'Mean {MAT_NAME} coverage (%)',
         'mature_length_pct_small': f'Mature small vessels (< {SMALL_VESSEL_UM} µm, % length)'}
MCOLS = {k: v for k, v in MCOLS.items() if k in MAT and MAT[k].notna().any()}
SP = MAT[MAT.specific]
Pm, Mm = SP[SP.condition == 'primary'], SP[SP.condition == 'metastasis']
fig, axs = plt.subplots(1, len(MCOLS), figsize=(3.6 * len(MCOLS), 3.8), squeeze=False)
for ax, (col, t) in zip(axs.flat, MCOLS.items()):
    boxes(ax, [Pm[col], Mm[col]], [f'P (n={Pm[col].notna().sum()})', f'M (n={Mm[col].notna().sum()})'], [COND_COL['primary'], COND_COL['metastasis']])
    pv = mannwhitneyu(Pm[col].dropna(), Mm[col].dropna()).pvalue if Pm[col].notna().sum() > 1 and Mm[col].notna().sum() > 1 else np.nan
    ax.set_title(t + (f'\np = {fmt_p(pv)}' if np.isfinite(pv) else ''), fontsize=10); ax.grid(axis='y', ls='--', color='0.85'); ax.set_axisbelow(True)
fig.suptitle(f'Vessel maturity: primary tumours (P) vs metastases (M); without samples in which {MAT_NAME} is not vessel-specific',
             fontweight='bold', fontsize=11)
fig.tight_layout(); save(fig, '7-5_maturity_primary_vs_metastasis')

### 5.3 Check of the classification: proliferating endothelial cells
Endothelial cells inside or within `ENDO_TOUCH_UM` of a piece, share with Ki67 z > `KI67_Z`, around immature vs mature
pieces. Sprouting, immature vessels are expected to have more Ki67⁺ endothelial cells. Left: the two values of each
sample. Middle: the per-sample values of all samples (box plots; grey lines join the two values of a sample), with the
Wilcoxon signed-rank test. Right: the difference immature − mature per sample (above 0 = more proliferation at
immature vessels), primary tumours vs metastases (Mann–Whitney). If most samples show no or the opposite difference,
check the coverage threshold and the QC figures before reading 5.4. Only the samples used in 5.3–5.6 (see 5.2).

In [ ]:
def cell_table(lab):
    """Cells near vessels with class, Ki67/HIF1A z-scores and the maturity data of their piece."""
    d = S[lab]; s = d['sample']
    C = pd.read_csv(d['dir'] + f'{s}_cells_vessels.csv', dtype={0: str}); C = C.set_index(C.columns[0]); C.index = C.index.astype(str)
    C = C.loc[C.index.intersection(CELLP[lab].index)].join(CELLP[lab])
    C['cls'] = C.cell_type.map(CT_MAP)
    a = ad.read_h5ad(d['base'] + 'adata_annotated.h5ad', backed='r')
    var = list(a.var_names); idx = a.obs_names.astype(str).get_indexer(C.index)
    for mk in ('HIF1A', 'Ki67'):
        if mk in var:
            col = a.X[:, var.index(mk)]; col = np.asarray(col.toarray() if hasattr(col, 'toarray') else col, dtype=float).ravel()
            C[mk] = np.where(idx >= 0, col[np.clip(idx, 0, None)], np.nan)
    a.file.close()
    p = PIECES[lab]
    C['width_um'] = p.width_um.reindex(C.piece).to_numpy()
    C['coverage'] = maturity_coverage(p).reindex(C.piece).to_numpy()     # coverage that decides the class
    C['border_px'] = p.border_px.reindex(C.piece).to_numpy()
    dens = p.local_vessel_area_pct.reindex(C.piece).to_numpy()
    med = np.nanmedian(np.repeat(p.local_vessel_area_pct.to_numpy(), np.maximum(1, np.round(p.length_um / 10).astype(int))))
    C['density'] = np.where(dens >= med, 'high', 'low')              # local vessel density: above / below the sample median
    C['dbin'] = pd.cut(C.dist_um, NEAR_BINS_UM, right=True, include_lowest=True,
                       labels=[f'{a_}–{b_} µm' for a_, b_ in zip(NEAR_BINS_UM[:-1], NEAR_BINS_UM[1:])])
    return C

def mclass(C, thr):
    return np.select([(C.border_px < MIN_BORDER_PX) | C.coverage.isna(), C.coverage >= thr], ['unclassified', 'mature'], 'immature')

CT = {lab: cell_table(lab) for lab in PIECES}
END = []
for lab, C in CT.items():
    e = C[(C.cls == 'Endothelial cells') & (C.dist_um <= ENDO_TOUCH_UM)]
    if 'Ki67' not in e or e.Ki67.isna().all() or not eligible(lab): continue
    k = mclass(e, MATURE_MIN)
    r = {c: (100 * (e.Ki67[k == c].dropna() > KI67_Z).mean() if e.Ki67[k == c].notna().sum() >= MIN_CELLS else np.nan) for c in ('immature', 'mature')}
    END.append(dict(label=lab, condition=S[lab]['condition'], **{f'ki67_endo_{c}_pct': v for c, v in r.items()},
                    n_immature=int((k == 'immature').sum()), n_mature=int((k == 'mature').sum())))
END = pd.DataFrame(END).set_index('label') if END else pd.DataFrame()

def paired_boxes(ax, a, b, conds, width=0.5):
    """Box plots of two paired series (one point per sample, grey line per sample, points coloured by condition)."""
    for x, v, c in ((0, a, CLS_PLOT['mature']), (1, b, CLS_PLOT['immature'])):
        q1, med, q3 = np.percentile(v, [25, 50, 75])
        ax.add_patch(plt.Rectangle((x - width / 2, q1), width, q3 - q1, facecolor=matplotlib.colors.to_rgba(c, 0.35), edgecolor=c, lw=1.6))
        ax.plot([x - width / 2, x + width / 2], [med, med], color=c, lw=2.4)
        ax.plot([x, x], [v.min(), q1], color=c, lw=1.4); ax.plot([x, x], [q3, v.max()], color=c, lw=1.4)
    for va, vb in zip(a, b):
        ax.plot([0.12, 0.88], [va, vb], color='0.7', lw=0.9, zorder=2)
    for x, v in ((0.12, a), (0.88, b)):
        ax.scatter(np.full(len(v), x), v, s=30, c=[COND_COL.get(c, '0.5') for c in conds], edgecolor='white', lw=0.8, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xlim(-0.6, 1.6)

if len(END):
    END['difference_pp'] = END.ki67_endo_immature_pct - END.ki67_endo_mature_pct
    ok = END.dropna(subset=['ki67_endo_immature_pct', 'ki67_endo_mature_pct'])
    ok = ok.reindex([l for l in S if l in ok.index])
    pw = wilcoxon(ok.ki67_endo_immature_pct, ok.ki67_endo_mature_pct).pvalue if len(ok) >= 6 else np.nan
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={'width_ratios': [2.4, 1, 1]})
    # A: per sample, mature vs immature
    x = np.arange(len(ok)); w = 0.38
    for dx, c_ in ((-w / 2, 'mature'), (w / 2, 'immature')):
        axs[0].bar(x + dx, ok[f'ki67_endo_{c_}_pct'], width=w - 0.04, color=CLS_PLOT[c_], label=c_)
    axs[0].set_xticks(x); axs[0].set_xticklabels(ok.index, rotation=60, ha='right', fontsize=9)
    axs[0].set_ylabel('Ki67⁺ endothelial cells (%)'); axs[0].set_title('Per sample', fontsize=11)
    top = 1.3 * max(ok.ki67_endo_mature_pct.max(), ok.ki67_endo_immature_pct.max(), 1)
    axs[0].set_ylim(0, top); axs[1].set_ylim(0, top)                   # same scale, starting at 0
    axs[0].legend(frameon=False, fontsize=9, loc='upper left', ncol=2,
                  title=f'vessel class ({MAT_NAME} coverage ≥ {MATURE_MIN:.0%} = mature)', title_fontsize=9)
    # B: all samples, paired
    paired_boxes(axs[1], ok.ki67_endo_mature_pct.to_numpy(), ok.ki67_endo_immature_pct.to_numpy(), ok.condition)
    axs[1].set_xticklabels(['mature', 'immature']); axs[1].set_ylabel('Ki67⁺ endothelial cells (%)')
    axs[1].set_title(f'All samples (n = {len(ok)})\nWilcoxon p = {fmt_p(pw)}', fontsize=11)
    # C: difference, primary vs metastasis
    dP, dM = ok.difference_pp[ok.condition == 'primary'], ok.difference_pp[ok.condition == 'metastasis']
    pmw = mannwhitneyu(dP, dM).pvalue if len(dP) > 1 and len(dM) > 1 else np.nan
    axs[2].axhline(0, color='k', lw=1)
    boxes(axs[2], [dP, dM], [f'P (n={len(dP)})', f'M (n={len(dM)})'], [COND_COL['primary'], COND_COL['metastasis']])
    axs[2].set_ylabel('immature − mature (percentage points)')
    axs[2].set_title(f'Difference (> 0: more at immature)\nP vs M: Mann–Whitney p = {fmt_p(pmw)}', fontsize=11)
    axs[1].legend(handles=[Line2D([0], [0], marker='o', ls='', color=c, label=k) for k, c in COND_COL.items() if k in set(ok.condition)],
                  frameon=False, fontsize=8, loc='upper right')
    for ax in axs:
        ax.grid(axis='y', ls='--', color='0.88'); ax.set_axisbelow(True)
    fig.suptitle('Proliferating (Ki67⁺) endothelial cells at immature vs mature vessels', fontweight='bold')
    fig.tight_layout(); save(fig, '7-5_endothelial_ki67_check')
    display(END.round(1)) if 'display' in globals() else print(END.round(1).to_string())

### 5.4 Tissue around immature vs mature vessels
For each sample: among the cells (without endothelial cells) whose nearest vessel piece is immature, the share of
HIF1A⁺ cells, Ki67⁺ cells and macrophages, compared with the cells around mature pieces. Shares are computed per stratum
(distance bin × low/high local vessel density) and averaged with the same weights (all cells of the stratum) for both
classes, so that a difference is not caused by the cells around one class lying farther from vessels or in sparser areas.
Strata with fewer than `MIN_STRATUM` cells in one class are left out. Effect = log2(share near immature / share near
mature); one point per sample. A value is only calculated if at least `MIN_POSITIVE` positive cells (HIF1A⁺, Ki67⁺ or
macrophages) lie around immature and around mature pieces in the strata used, because a ratio of a few cells is mostly
noise; the table gives these numbers (around immature / around mature). Only the samples used in 5.3–5.6 (see 5.2).

Figure, one row per cell group. Left: the adjusted share near mature and near immature pieces for each sample (n.e. =
not evaluated). Middle: the same values for all samples (box plots; grey lines join the two values of a sample), with
the Wilcoxon signed-rank test of the log2 ratio (is it consistently ≠ 0?). Right: the log2 ratio of the primary tumours
and the metastases (Mann–Whitney). P-values are not corrected for multiple testing.

In [ ]:
GROUPS = {'HIF1A⁺ cells': lambda C: (C.HIF1A > HIF1A_Z).where(C.HIF1A.notna()) if 'HIF1A' in C else None,   # NaN: no z-score
          'Ki67⁺ cells': lambda C: (C.Ki67 > KI67_Z).where(C.Ki67.notna()) if 'Ki67' in C else None,
          'Macrophages': lambda C: C.cls.isin(MACROPHAGE_CLASSES)}

def adjusted_effect(C, flag, cls):
    """Stratum-standardised shares near immature and mature pieces, log2 ratio, number of cells and of positive cells."""
    D = pd.DataFrame({'flag': flag.astype(float), 'cls': cls, 'st': C.dbin.astype(str) + ' | ' + C.density})
    D = D[D.cls.isin(['immature', 'mature']) & flag.notna()]
    g = D.groupby(['st', 'cls']).flag.agg(['sum', 'count']).unstack('cls')
    if g.empty or 'immature' not in g['count'] or 'mature' not in g['count']:
        return np.nan, np.nan, np.nan, 0, 0, 0
    g = g[(g['count']['immature'] >= MIN_STRATUM) & (g['count']['mature'] >= MIN_STRATUM)]
    if g.empty:
        return np.nan, np.nan, np.nan, 0, 0, 0
    w = g['count'].sum(axis=1)
    pi = float((g['sum']['immature'] / g['count']['immature'] * w).sum() / w.sum())
    pm = float((g['sum']['mature'] / g['count']['mature'] * w).sum() / w.sum())
    eps = 0.5 / w.sum()                                              # avoids log(0) for rare groups
    return (np.log2((pi + eps) / (pm + eps)), 100 * pi, 100 * pm, int(w.sum()),
            int(g['sum']['immature'].sum()), int(g['sum']['mature'].sum()))

def effects(thr, small_only=False):
    rows = []
    for lab, C in CT.items():
        C = C[C.cls.notna() & (C.cls != 'Endothelial cells')]
        if small_only: C = C[C.width_um < SMALL_VESSEL_UM]
        cls = mclass(C, thr)
        ni, nm = n_pieces(PIECES[lab], thr, small_only)
        why = '' if specific(lab) else f'{MAT_NAME} not vessel-specific'
        if not why and min(ni, nm) < MIN_PIECES: why = f'< {MIN_PIECES} immature or mature pieces'
        for gname, fn in GROUPS.items():
            flag = fn(C)
            if flag is None or flag.isna().all(): continue
            e, pi, pm, n, ki, km = adjusted_effect(C, flag, cls)
            note = why or ('' if min(ki, km) >= MIN_POSITIVE else f'< {MIN_POSITIVE} positive cells')
            rows.append(dict(label=lab, condition=S[lab]['condition'], group=gname, log2_ratio=np.nan if note else e,
                             pct_near_immature=pi, pct_near_mature=pm, n_cells=n, n_pos_immature=ki, n_pos_mature=km,
                             n_immature_pieces=ni, n_mature_pieces=nm, sample_not_used=why, not_evaluated=note))
    return pd.DataFrame(rows)

def summarise(EF):
    out = []
    for gname in GROUPS:
        e = EF[EF.group == gname].dropna(subset=['log2_ratio'])
        pw = wilcoxon(e.log2_ratio).pvalue if len(e) >= 6 and (e.log2_ratio != 0).any() else np.nan
        ep, em = e[e.condition == 'primary'].log2_ratio, e[e.condition == 'metastasis'].log2_ratio
        pmw = mannwhitneyu(ep, em).pvalue if len(ep) > 1 and len(em) > 1 else np.nan
        out.append(dict(group=gname, n_samples=len(e), median_log2_ratio=e.log2_ratio.median() if len(e) else np.nan,
                        samples_above_0=int((e.log2_ratio > 0).sum()), wilcoxon_p=pw,
                        median_P=ep.median() if len(ep) else np.nan, median_M=em.median() if len(em) else np.nan, mannwhitney_P_vs_M_p=pmw))
    return pd.DataFrame(out).set_index('group')

EF = effects(MATURE_MIN)
EF.to_csv(FIG_DIR + 'maturity_effects.csv', index=False)
if len(EF):                                                          # positive cells around immature / mature pieces
    few_pos = (EF.sample_not_used == '') & (EF.not_evaluated != '')
    CNT = EF.assign(cells=EF.n_pos_immature.astype(str) + ' / ' + EF.n_pos_mature.astype(str)
                    + np.where(few_pos, ' (too few)', '')).pivot(index='label', columns='group', values='cells')
    CNT['sample not used'] = EF.groupby('label').sample_not_used.first()
    CNT = CNT.reindex([l for l in S if l in CNT.index])
    print('positive cells around immature / mature pieces (strata used):')
    display(CNT) if 'display' in globals() else print(CNT.to_string())
SUMM = summarise(EF)
display(SUMM.round(3)) if 'display' in globals() else print(SUMM.round(3).to_string())

if len(EF):
    labs = [l for l in S if l in set(EF.label) and EF.loc[EF.label == l, 'sample_not_used'].iloc[0] == '']
    notused = [l for l in S if l in set(EF.label) and l not in labs]
    groups = [g for g in GROUPS if (EF.group == g).any()]
    fig, axs = plt.subplots(len(groups), 3, figsize=(16, 4.4 * len(groups)), gridspec_kw={'width_ratios': [2.4, 1, 1]}, squeeze=False)
    for row, gname in zip(axs, groups):
        e = EF[EF.group == gname].set_index('label').reindex(labs)
        ev = e[e.log2_ratio.notna()]
        r = SUMM.loc[gname]
        top = 1.3 * max(np.nanmax(ev[['pct_near_mature', 'pct_near_immature']].to_numpy()) if len(ev) else 0, 0.1)
        # left: per sample
        x, w = np.arange(len(labs)), 0.38
        for dx, c_ in ((-w / 2, 'mature'), (w / 2, 'immature')):
            row[0].bar(x + dx, e[f'pct_near_{c_}'].where(e.log2_ratio.notna()), width=w - 0.04, color=CLS_PLOT[c_], label=f'near {c_}')
        for xi, v_ in zip(x, e.log2_ratio):
            if not np.isfinite(v_): row[0].text(xi, top * 0.02, 'n.e.', ha='center', va='bottom', fontsize=8, color='0.5')
        row[0].set_xticks(x); row[0].set_xticklabels(labs, rotation=60, ha='right', fontsize=9)
        row[0].set_ylim(0, top); row[0].set_ylabel(f'{gname} (% of cells)')
        row[0].set_title(f'{gname}: per sample (n.e. = not evaluated)', fontsize=11, loc='left', fontweight='bold')
        row[0].legend(frameon=False, fontsize=9, loc='upper left', ncol=2)
        # middle: all samples, paired
        if len(ev) >= 2:
            paired_boxes(row[1], ev.pct_near_mature.to_numpy(), ev.pct_near_immature.to_numpy(), ev.condition)
            row[1].set_xticklabels(['near mature', 'near immature'])
            row[1].legend(handles=[Line2D([0], [0], marker='o', ls='', color=c, label=k) for k, c in COND_COL.items() if k in set(ev.condition)],
                          frameon=False, fontsize=8, loc='upper right')
        row[1].set_ylim(0, top); row[1].set_ylabel('% of cells')
        row[1].set_title(f'All samples (n = {len(ev)})\nWilcoxon p = {fmt_p(r.wilcoxon_p)}', fontsize=11)
        # right: ratio, primary vs metastasis
        lP, lM = ev.log2_ratio[ev.condition == 'primary'], ev.log2_ratio[ev.condition == 'metastasis']
        row[2].axhline(0, color='k', lw=1)
        boxes(row[2], [lP, lM], [f'P (n={len(lP)})', f'M (n={len(lM)})'], [COND_COL['primary'], COND_COL['metastasis']])
        row[2].set_ylabel('log2(near immature / near mature)')
        row[2].set_title(f'Ratio (> 0: more near immature)\nP vs M: Mann–Whitney p = {fmt_p(r.mannwhitney_P_vs_M_p)}', fontsize=11)
        for ax in row:
            ax.grid(axis='y', ls='--', color='0.88'); ax.set_axisbelow(True)
    note = f'\nnot used (see 5.2): {", ".join(notused)}' if notused else ''
    fig.suptitle(f'Tissue around immature vs mature vessels ({MAT_NAME} coverage ≥ {MATURE_MIN:.0%} = mature; '
                 f'shares adjusted for distance and local vessel density){note}', fontweight='bold', fontsize=11)
    fig.tight_layout(); save(fig, '7-5_maturity_effects')

### 5.5 Share of positive cells vs distance, immature vs mature
Share of HIF1A⁺ cells, Ki67⁺ cells and macrophages per distance bin around mature (gold) and immature (blue) pieces:
box plots of the samples (one point per sample). The p above each pair is the Wilcoxon signed-rank test, paired by
sample (not corrected for multiple testing). Not adjusted for local vessel density, unlike 5.4. Only the samples used
in 5.3–5.6 (see 5.2).

In [ ]:
binlabs = [f'{a_}–{b_} µm' for a_, b_ in zip(NEAR_BINS_UM[:-1], NEAR_BINS_UM[1:])]
CUR = []
for lab, C in CT.items():
    if not eligible(lab): continue
    C = C[C.cls.notna() & (C.cls != 'Endothelial cells')]; cls = mclass(C, MATURE_MIN)
    for gname, fn in GROUPS.items():
        flag = fn(C)
        if flag is None or flag.isna().all(): continue
        for c_ in ('immature', 'mature'):
            sel = cls == c_
            g = flag[sel].astype(float).groupby(C.dbin[sel], observed=False).agg(['mean', 'count'])
            for b_, r in g.iterrows():
                if r['count'] >= MIN_CELLS: CUR.append(dict(label=lab, group=gname, cls=c_, bin=str(b_), pct=100 * r['mean']))
CUR = pd.DataFrame(CUR)
if len(CUR):
    groups = [g for g in GROUPS if (CUR.group == g).any()]
    fig, axs = plt.subplots(1, len(groups), figsize=(5.4 * len(groups), 4.6), squeeze=False)
    for ax, gname in zip(axs.flat, groups):
        Q = CUR[CUR.group == gname].pivot_table(index='label', columns=['bin', 'cls'], values='pct')
        data, pos, cols = [], [], []
        for b, bl in enumerate(binlabs):
            for dx, c_ in ((-0.21, 'mature'), (0.21, 'immature')):
                data.append(Q[(bl, c_)].dropna().to_numpy() if (bl, c_) in Q else np.array([]))
                pos.append(b + dx); cols.append(CLS_PLOT[c_])
        boxes(ax, data, [''] * len(data), cols, positions=pos, width=0.36)
        top = max([v.max() for v in data if len(v)] + [0.1])
        for b, bl in enumerate(binlabs):                              # paired test per distance bin
            if (bl, 'mature') in Q and (bl, 'immature') in Q:
                pr = Q[[(bl, 'mature'), (bl, 'immature')]].dropna()
                ok_ = len(pr) >= 6 and (pr.iloc[:, 0] != pr.iloc[:, 1]).any()
                p_ = wilcoxon(pr.iloc[:, 1], pr.iloc[:, 0]).pvalue if ok_ else np.nan
                ax.text(b, top * 1.04, f'p = {fmt_p(p_)}', ha='center', va='bottom', fontsize=8.5, color='0.3')
        ax.set_ylim(0, top * 1.16)
        ax.set_xticks(range(len(binlabs))); ax.set_xticklabels(binlabs); ax.set_xlim(-0.6, len(binlabs) - 0.4)
        ax.set_title(gname, fontsize=11, fontweight='bold'); ax.set_ylabel('% of cells'); ax.set_xlabel('distance to the nearest vessel')
        ax.grid(axis='y', ls='--', color='0.88'); ax.set_axisbelow(True)
    fig.suptitle('Share of positive cells by distance, around mature vs immature vessels (one point per sample; '
                 'p: Wilcoxon, paired by sample)', fontweight='bold', fontsize=11, y=0.99)
    fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=matplotlib.colors.to_rgba(CLS_PLOT[c_], 0.5), ec=CLS_PLOT[c_], label=f'near {c_}')
                        for c_ in ('mature', 'immature')], frameon=False, fontsize=9, loc='upper center', bbox_to_anchor=(0.5, 0.945), ncol=2)
    fig.tight_layout(rect=[0, 0, 1, 0.92]); save(fig, '7-5_maturity_vs_distance')

### 5.6 Sensitivity
The median effect (log2 ratio) and the Wilcoxon p for other αSMA coverage thresholds, and for small vessels only (mean width
< `SMALL_VESSEL_UM`; the cells whose nearest piece is a small vessel). A conclusion should not depend on one threshold.

In [ ]:
SENS = []
for thr in SENSITIVITY_MIN:
    s_ = summarise(effects(thr)); s_['threshold'] = thr; s_['vessels'] = 'all'; SENS.append(s_)
s_ = summarise(effects(MATURE_MIN, small_only=True)); s_['threshold'] = MATURE_MIN; s_['vessels'] = f'< {SMALL_VESSEL_UM} µm'; SENS.append(s_)
SENS = pd.concat(SENS).reset_index().set_index(['vessels', 'threshold', 'group'])
SENS.to_csv(FIG_DIR + 'maturity_sensitivity.csv')
display(SENS[['n_samples', 'median_log2_ratio', 'samples_above_0', 'wilcoxon_p', 'mannwhitney_P_vs_M_p']].round(3)) if 'display' in globals() \
    else print(SENS[['n_samples', 'median_log2_ratio', 'samples_above_0', 'wilcoxon_p', 'mannwhitney_P_vs_M_p']].round(3).to_string())

### 5.7 The features of 4.1 for mature and immature vessels, primary tumours vs metastases
The features of 4.1 are calculated again for every sample, once with only the **mature** vessels (αSMA coverage ≥
`MATURE_MIN`) and once with only the **immature** vessels (all other vessel pieces), as if the other vessels were not
there:
- **Vessel area, vessel length density**: the mature or the immature pieces, per mm² of tumour region.
- **Vessel width**: median of the widths of the pieces, each piece weighted by its area.
- **Distance to the nearest vessel, tissue > `POOR_SUPPLY_UM`**: distance of every point of the tissue (2 µm grid,
  without the vessels) to the nearest mature or the nearest immature vessel.
- **Heterogeneity**: coefficient of variation of the mature or the immature vessel area across the `GRID_UM` blocks.
- **Pericyte- and macrophage-covered vessel area**: share of the mature or the immature vessel area in pieces with at
  least one mural cell or macrophage within `NEAR_VESSEL_UM` (each cell counts for its nearest vessel piece). This is per
  piece, so the values are lower than in part 3, where a whole vessel object counted as covered.
- **Ki67⁺ endothelial cells**: endothelial cells within `ENDO_TOUCH_UM` of a mature or an immature piece (as in 5.3).
- **Distance of HIF1A⁺ cells, Ki67⁺ cells and macrophages**: median distance to the nearest mature or the nearest
  immature vessel, relative to the median distance of all cells (endothelial cells left out; 2 µm grid).
- **Macrophages per mm vessel**: macrophages within `NEAR_VESSEL_UM` of a mature or an immature piece, per mm of mature or
  immature vessel length.

Only the samples used in 5.3–5.6 (see 5.2); the recurrence is left out, as in 4.1. The table is written to
`features_by_vessel_class.csv`, the tests to `features_by_vessel_class_tests.csv`.

Figure: one panel per feature, four groups (mature / immature vessels of the primary tumours and of the metastases), as
box plots or violin plots (`GROUP_PLOT`). Points: samples (blue = primary, orange = metastasis); grey lines join the
mature and the immature value of the same sample. Brackets: mature vs immature within the primary tumours and within
the metastases (Wilcoxon signed-rank test, paired by sample). Title: primary tumours vs metastases, for the mature and
for the immature vessels (Mann–Whitney U test). P-values are not corrected for multiple testing.

In [ ]:
GROUP_PLOT = 'box'      # 'box' or 'violin'
VCLASSES = ('mature', 'immature')

def class_maps(lab):
    """Mature and immature vessel masks on the 2 µm grid, vessel pixels of each class per GRID_UM block, region, tissue."""
    d = S[lab]; s = d['sample']
    labels = tifffile.imread(d['dir'] + f'{s}_vessel_labels.tif'); H, W = labels.shape
    m = np.load(d['base'] + 'masks_lowres.npz'); ds = int(m['bg_ds'])
    region = (m['region_lr'] if 'region_lr' in m else m['tissue_lr']).astype(bool)
    region &= ~annotation_mask(ARTEFACT_GEOJSON.format(s=s), region.shape, ds)
    h, w = region.shape
    p = PIECES[lab]
    mature_keys = np.sort(p.index[classify(p, MATURE_MIN) == 'mature'].to_numpy(np.int64))
    seg = int(round(PIECE_UM / PIXEL_SIZE_UM)); ngx = -(-W // seg); vmax = int(labels.max()) + 1   # as in 5.1
    step = ds * PIXEL_SIZE_UM; b = max(1, int(round(GRID_UM / step))); bf = b * ds                  # as in part 2
    reg_b = block_sum(region.astype(np.float32), b); nby, nbx = reg_b.shape
    lr = {c: np.zeros((h, w), bool) for c in VCLASSES}
    cnt = {c: np.zeros(nby * nbx, np.float64) for c in VCLASSES}
    for y0 in range(0, H, TILE_PX):
        y1 = min(y0 + TILE_PX, H)
        yy, xx = np.nonzero(labels[y0:y1]); yy = yy + y0
        if not len(yy): continue
        key = ((yy // seg).astype(np.int64) * ngx + xx // seg) * vmax + labels[yy, xx]
        mat = np.isin(key, mature_keys)
        for c, sel in (('mature', mat), ('immature', ~mat)):          # everything that is not mature is immature
            ly, lx = yy[sel] // ds, xx[sel] // ds; ok = (ly < h) & (lx < w)
            lr[c][ly[ok], lx[ok]] = True
            by, bx = yy[sel] // bf, xx[sel] // bf; ok = (by < nby) & (bx < nbx)
            cnt[c] += np.bincount(by[ok] * nbx + bx[ok], minlength=nby * nbx)
    del labels
    return dict(lr=lr, cnt={c: v.reshape(nby, nbx) for c, v in cnt.items()}, region=region,
                tissue=region & ~(lr['mature'] | lr['immature']), reg_b=reg_b, b=b, ds=ds, step=step)

def all_cells(lab):
    """All 03c cells (without the artefact areas) with class, Ki67 / HIF1A z-scores and their nearest piece (5.1)."""
    d = S[lab]; s = d['sample']
    C = pd.read_csv(d['dir'] + f'{s}_cells_vessels.csv', dtype={0: str}); C = C.set_index(C.columns[0]); C.index = C.index.astype(str)
    if 'in_artefact_area' in C: C = C[~C.in_artefact_area.astype(bool)]
    C['cls'] = C.cell_type.map(CT_MAP)
    a = ad.read_h5ad(d['base'] + 'adata_annotated.h5ad', backed='r')
    var = list(a.var_names); idx = a.obs_names.astype(str).get_indexer(C.index)
    for mk in ('HIF1A', 'Ki67'):
        if mk in var:
            col = a.X[:, var.index(mk)]; col = np.asarray(col.toarray() if hasattr(col, 'toarray') else col, dtype=float).ravel()
            C[mk] = np.where(idx >= 0, col[np.clip(idx, 0, None)], np.nan)
    a.file.close()
    return C.join(CELLP[lab][['dist_um', 'piece']], how='left')       # cells within NEAR_BINS_UM[-1] of a vessel

def class_features(lab):
    """The features of 4.1 with only the mature and with only the immature vessels of one sample."""
    G, C, p = class_maps(lab), all_cells(lab), PIECES[lab]
    pc = classify(p, MATURE_MIN).replace('unclassified', 'immature')
    region_mm2 = F.at[lab, 'region_mm2']
    yi = np.clip((C.y.to_numpy() // G['ds']).astype(int), 0, G['region'].shape[0] - 1)
    xi = np.clip((C.x.to_numpy() // G['ds']).astype(int), 0, G['region'].shape[1] - 1)
    piece = C.piece.fillna(-1).astype(np.int64)
    C['piece_cls'] = pc.reindex(piece).to_numpy()                     # NaN: no vessel within NEAR_BINS_UM[-1]
    other = C.cls != 'Endothelial cells'
    flags = {'hif1a_pos': (C.HIF1A > HIF1A_Z) if 'HIF1A' in C else None, 'ki67_pos': (C.Ki67 > KI67_Z) if 'Ki67' in C else None,
             'macrophage': C.cls.isin(MACROPHAGE_CLASSES)}
    rows = []
    for c in VCLASSES:
        q = p[pc == c]; area, length = q.area_um2.sum(), q.length_um.sum()
        r = dict(label=lab, condition=S[lab]['condition'], vessel_class=c, n_pieces=len(q),
                 vessel_area_pct=100 * area / (region_mm2 * 1e6), vessel_length_mm_per_mm2=length / 1000 / region_mm2)
        if len(q):
            o = np.argsort(q.width_um.to_numpy()); cw = np.cumsum(q.area_um2.to_numpy()[o]) / max(area, 1e-9)
            r['width_um_median'] = q.width_um.to_numpy()[o][min(np.searchsorted(cw, 0.5), len(o) - 1)]
        # distances to the nearest vessel of this class
        if G['lr'][c].any():
            dist = ndi.distance_transform_edt(~G['lr'][c]) * G['step']
            dt = dist[G['tissue']]
            if len(dt):
                r['dist_to_vessel_um_median'] = np.median(dt); r['poorly_supplied_pct'] = 100 * (dt > POOR_SUPPLY_UM).mean()
            dcell = pd.Series(dist[yi, xi], index=C.index); del dist
            med_all = np.median(dcell[other]) if other.any() else np.nan
            for name, flag in flags.items():
                if flag is None: continue
                dp = dcell[other & flag]
                if len(dp) >= MIN_CELLS and med_all > 0: r[f'{name}_rel_dist'] = np.median(dp) / med_all
        # heterogeneity
        use = G['reg_b'] >= MIN_BLOCK_COVER * G['b'] ** 2
        fv = (100 * G['cnt'][c] / np.maximum(G['reg_b'] * G['ds'] ** 2, 1))[use]
        r['block_cv'] = np.std(fv) / np.mean(fv) if len(fv) and np.mean(fv) > 0 else np.nan
        # cells next to the pieces of this class
        nearc = (C.piece_cls == c) & (C.dist_um <= NEAR_VESSEL_UM)
        for name, sel in (('pericyte', C.cls.isin(MURAL_CLASSES)), ('macrophage', C.cls.isin(MACROPHAGE_CLASSES))):
            covered = set(piece[nearc & sel])
            r[f'{name}_covered_area_pct'] = 100 * q.area_um2[q.index.isin(covered)].sum() / area if area > 0 else np.nan
        mac = C.cls.isin(MACROPHAGE_CLASSES)
        r['perivascular_macrophages_per_mm_vessel'] = (nearc & mac).sum() / (length / 1000) if length > 0 and mac.sum() >= MIN_CELLS else np.nan
        endo = C[(C.cls == 'Endothelial cells') & (C.piece_cls == c) & (C.dist_um <= ENDO_TOUCH_UM)]
        r['ki67_endothelial_pct'] = 100 * (endo.Ki67 > KI67_Z).mean() if 'Ki67' in endo and endo.Ki67.notna().sum() >= MIN_CELLS else np.nan
        rows.append(r)
    return rows

CF57 = []
for lab in S:
    if S[lab]['condition'] not in ('primary', 'metastasis'):
        print(f'{lab}: {S[lab]["condition"]}, left out'); continue
    if lab not in PIECES or not eligible(lab):
        print(f'{lab}: not used in 5.3–5.6, left out'); continue
    CF57 += class_features(lab); print(f'{lab}: done')
CF57 = pd.DataFrame(CF57)
CF57.to_csv(FIG_DIR + 'features_by_vessel_class.csv', index=False)

In [ ]:
GROUPS57 = [('primary', 'mature'), ('primary', 'immature'), ('metastasis', 'mature'), ('metastasis', 'immature')]
POS57 = [0, 1, 2.7, 3.7]
FEAT57 = {k: v for k, v in {**VESSEL_FEATURES, **CELL_FEATURES}.items() if k in CF57 and CF57[k].notna().any()}

def group_plot(ax, col):
    """Four groups (mature / immature x primary / metastasis) of one feature; returns the p-values."""
    Wd = CF57.pivot_table(index=['label', 'condition'], columns='vessel_class', values=col).reset_index()
    for c_ in VCLASSES:
        if c_ not in Wd: Wd[c_] = np.nan
    vals = [Wd.loc[Wd.condition == cond, c_].dropna().to_numpy() for cond, c_ in GROUPS57]
    for (cond, c_), x, v in zip(GROUPS57, POS57, vals):
        if not len(v): continue
        cc = CLS_PLOT[c_]
        if GROUP_PLOT == 'violin' and len(v) >= 3 and np.ptp(v) > 0:
            vp = ax.violinplot(v, positions=[x], widths=0.8, showextrema=False)
            for b_ in vp['bodies']:
                b_.set_facecolor(matplotlib.colors.to_rgba(cc, 0.35)); b_.set_edgecolor(cc); b_.set_linewidth(1.2); b_.set_alpha(None)
            ax.plot([x - 0.2, x + 0.2], [np.median(v)] * 2, color=cc, lw=2.4)
        else:
            q1, med, q3 = np.percentile(v, [25, 50, 75])
            ax.add_patch(plt.Rectangle((x - 0.3, q1), 0.6, q3 - q1, facecolor=matplotlib.colors.to_rgba(cc, 0.35), edgecolor=cc, lw=1.6))
            ax.plot([x - 0.3, x + 0.3], [med, med], color=cc, lw=2.4)
            ax.plot([x, x], [v.min(), q1], color=cc, lw=1.4); ax.plot([x, x], [q3, v.max()], color=cc, lw=1.4)
    allv = np.concatenate([v for v in vals if len(v)]) if any(len(v) for v in vals) else np.array([0.0])
    top, bot = allv.max(), min(allv.min(), 0); span = (top - bot) or 1
    out = {}
    for cond, x0, x1 in (('primary', 0, 1), ('metastasis', 2.7, 3.7)):
        Wc = Wd[Wd.condition == cond]
        for _, r in Wc.iterrows():                                     # one grey line per sample
            if np.isfinite(r['mature']) and np.isfinite(r['immature']):
                ax.plot([x0 + 0.2, x1 - 0.2], [r['mature'], r['immature']], color='0.7', lw=0.9, zorder=2)
        for c_, x in (('mature', x0 + 0.2), ('immature', x1 - 0.2)):
            v = Wc[c_].dropna()
            ax.scatter(np.full(len(v), x), v, s=26, color=COND_COL[cond], edgecolor='white', lw=0.7, zorder=3)
        pr = Wc[['mature', 'immature']].dropna()                       # mature vs immature, paired by sample
        pw = wilcoxon(pr.mature, pr.immature).pvalue if len(pr) >= 6 and (pr.mature != pr.immature).any() else np.nan
        out[f'{cond}_mature_vs_immature_p'] = pw
        y = top + 0.07 * span
        ax.plot([x0, x0, x1, x1], [y - 0.025 * span, y, y, y - 0.025 * span], color='0.3', lw=1)
        ax.text((x0 + x1) / 2, y + 0.01 * span, f'p = {fmt_p(pw)}', ha='center', va='bottom', fontsize=8.5)
    for c_ in VCLASSES:                                                # primary vs metastasis, per vessel class
        a_, b_ = vals[GROUPS57.index(('primary', c_))], vals[GROUPS57.index(('metastasis', c_))]
        out[f'{c_}_P_vs_M_p'] = mannwhitneyu(a_, b_).pvalue if len(a_) > 1 and len(b_) > 1 else np.nan
    if col.endswith('_rel_dist'): ax.axhline(1, color='k', lw=1, ls='--')   # 1 = same distance as the average cell
    ax.set_ylim(bot if bot < 0 else 0, top + 0.22 * span)
    ax.set_xticks(POS57); ax.set_xticklabels([f'{c_}\n{cond[0].upper()} (n={len(v)})' for (cond, c_), v in zip(GROUPS57, vals)], fontsize=8.5)
    ax.set_xlim(-0.6, 4.3); ax.grid(axis='y', ls='--', color='0.88'); ax.set_axisbelow(True)
    for (cond, c_), v in zip(GROUPS57, vals):
        out[f'median_{cond}_{c_}'] = np.median(v) if len(v) else np.nan
    return out

if len(CF57) and FEAT57:
    n = len(FEAT57); cols = 4; rws = -(-n // cols)
    fig, axs = plt.subplots(rws, cols, figsize=(4.3 * cols, 4.4 * rws), squeeze=False)
    TESTS = []
    for ax, (col, t) in zip(axs.flat, FEAT57.items()):
        r = group_plot(ax, col)
        ax.set_title(f"{t}\nP vs M: mature p = {fmt_p(r['mature_P_vs_M_p'])}, immature p = {fmt_p(r['immature_P_vs_M_p'])}", fontsize=9.5)
        TESTS.append(dict(feature=col, **r))
    for ax in list(axs.flat)[n:]: ax.axis('off')
    fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=matplotlib.colors.to_rgba(CLS_PLOT[c_], 0.35), ec=CLS_PLOT[c_],
                                      label=f'{c_} vessels' + (f' ({MAT_NAME} ≥ {MATURE_MIN:.0%})' if c_ == 'mature' else ''))
                        for c_ in VCLASSES]
               + [Line2D([0], [0], marker='o', ls='', color=COND_COL[k], label=k) for k in ('primary', 'metastasis')],
               frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.suptitle('Features of 4.1 for mature and immature vessels: primary tumours (P) vs metastases (M)\n'
                 'brackets: mature vs immature (Wilcoxon, paired by sample)', fontweight='bold', y=1.035)
    fig.tight_layout(rect=[0, 0, 1, 0.975]); fig.subplots_adjust(top=0.93); save(fig, '7-5_features_mature_vs_immature_P_vs_M')
    TESTS = pd.DataFrame(TESTS).set_index('feature')
    TESTS.to_csv(FIG_DIR + 'features_by_vessel_class_tests.csv')
    display(TESTS.round(3)) if 'display' in globals() else print(TESTS.round(3).to_string())

In [ ]:
def two_groups(ax, a, b, conds):
    """Mature (a) vs immature (b) of the same samples: box or violin plot, a grey line per sample, points by condition."""
    for x, v, c_ in ((0, a, 'mature'), (1, b, 'immature')):
        cc = CLS_PLOT[c_]
        if GROUP_PLOT == 'violin' and len(v) >= 3 and np.ptp(v) > 0:
            vp = ax.violinplot(v, positions=[x], widths=0.7, showextrema=False)
            for b_ in vp['bodies']:
                b_.set_facecolor(matplotlib.colors.to_rgba(cc, 0.35)); b_.set_edgecolor(cc); b_.set_linewidth(1.2); b_.set_alpha(None)
            ax.plot([x - 0.18, x + 0.18], [np.median(v)] * 2, color=cc, lw=2.4)
        else:
            q1, med, q3 = np.percentile(v, [25, 50, 75])
            ax.add_patch(plt.Rectangle((x - 0.25, q1), 0.5, q3 - q1, facecolor=matplotlib.colors.to_rgba(cc, 0.35), edgecolor=cc, lw=1.6))
            ax.plot([x - 0.25, x + 0.25], [med, med], color=cc, lw=2.4)
            ax.plot([x, x], [v.min(), q1], color=cc, lw=1.4); ax.plot([x, x], [q3, v.max()], color=cc, lw=1.4)
    for va, vb in zip(a, b):
        ax.plot([0.12, 0.88], [va, vb], color='0.7', lw=0.9, zorder=2)
    for x, v in ((0.12, a), (0.88, b)):
        ax.scatter(np.full(len(v), x), v, s=28, c=[COND_COL.get(c, '0.5') for c in conds], edgecolor='white', lw=0.7, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['mature', 'immature']); ax.set_xlim(-0.6, 1.6)

if len(CF57) and FEAT57:
    n = len(FEAT57); cols = 4; rws = -(-n // cols)
    fig, axs = plt.subplots(rws, cols, figsize=(3.6 * cols, 4.0 * rws), squeeze=False)
    TESTS_MI = []
    for ax, (col, t) in zip(axs.flat, FEAT57.items()):
        Wd = CF57.pivot_table(index=['label', 'condition'], columns='vessel_class', values=col).reset_index()
        for c_ in VCLASSES:
            if c_ not in Wd: Wd[c_] = np.nan
        pr = Wd.dropna(subset=list(VCLASSES))                           # samples with both values
        pw = wilcoxon(pr.mature, pr.immature).pvalue if len(pr) >= 6 and (pr.mature != pr.immature).any() else np.nan
        if len(pr):
            two_groups(ax, pr.mature.to_numpy(), pr.immature.to_numpy(), pr.condition)
            lo, hi = min(pr[list(VCLASSES)].min().min(), 0), pr[list(VCLASSES)].max().max()
            ax.set_ylim(lo, hi + 0.1 * ((hi - lo) or 1))
        if col.endswith('_rel_dist'): ax.axhline(1, color='k', lw=1, ls='--')   # 1 = same distance as the average cell
        ax.set_title(f'{t}\nn = {len(pr)}, Wilcoxon p = {fmt_p(pw)}', fontsize=9.5)
        ax.grid(axis='y', ls='--', color='0.88'); ax.set_axisbelow(True)
        TESTS_MI.append(dict(feature=col, n_samples=len(pr), median_mature=pr.mature.median(), median_immature=pr.immature.median(),
                             samples_immature_higher=int((pr.immature > pr.mature).sum()), wilcoxon_p=pw))
    for ax in list(axs.flat)[n:]: ax.axis('off')
    fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=matplotlib.colors.to_rgba(CLS_PLOT[c_], 0.35), ec=CLS_PLOT[c_],
                                      label=f'{c_} vessels' + (f' ({MAT_NAME} ≥ {MATURE_MIN:.0%})' if c_ == 'mature' else ''))
                        for c_ in VCLASSES]
               + [Line2D([0], [0], marker='o', ls='', color=COND_COL[k], label=k) for k in ('primary', 'metastasis')],
               frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.suptitle('Features of 4.1: mature vs immature vessels, primary tumours and metastases together\n'
                 'p: Wilcoxon, paired by sample', fontweight='bold', y=1.035)
    fig.tight_layout(rect=[0, 0, 1, 0.975]); fig.subplots_adjust(top=0.93); save(fig, '7-5_features_mature_vs_immature')
    TESTS_MI = pd.DataFrame(TESTS_MI).set_index('feature')
    TESTS_MI.to_csv(FIG_DIR + 'features_mature_vs_immature_tests.csv')
    display(TESTS_MI.round(3)) if 'display' in globals() else print(TESTS_MI.round(3).to_string())